## [L&E variable replace 코드 수정에 대한 근거 마련을 위한 EDA] + [Prototype]
(2024/12/10)
### EDA
#### 목적
- 기존 로직 : j=0…max_mask_cnt_per_span[i] 까지 현재 span을 j개 mask로 대체하고 candidate generation을 각각 하고 있었음
- candidate generation을 max_mask_cnt_per_span[i] 인 경우로 한번만 하고, 각 j=1,2,...,max_mask_cnt_per_span[i] 경우마다 beam search 만 따로 할지 (beam search 할 때 j=max_mask_cnt_per_span[i]일 때 생성한 candidate을 가져다 씀)
#### 확인한 내용
- 특정 span을 j개 mask로 대체했을 때, 그 중 첫번째 mask token에 대한 top k candidate이 j가 1,2,3.. 으로 늘어남에 따라 달라지는지 체크
- j는 최대 3으로 설정 / 문장 내 첫번째 span에 대해서 확인 / top k는 5로 설정
#### 결론
j=1,2,3일 때의 top5 candidate이 완전히 동일한 경우는 34% 정도. (309, 905), top5 중 적어도 4개씩은 candidate이 동일한 경우는 96% (869/905)

In [1]:
import re
from collections import defaultdict
import json 
from typing import List, Tuple

from torch.utils.data import DataLoader,Dataset
import transformers
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import torch
import numpy as np
import pandas as pd
import wandb

from new_module.new_decode_utils import get_beam_hypotheses_v0
import new_module.losses as lossbuilder

def check_equality(x):
    if torch.equal(x[0],x[1]) and torch.equal(x[1],x[2]) and torch.equal(x[0],x[2]):
        return True
    else:
        return False

def check_loose_equality(x, min_overlap=2):
    def loosely_equal(tensor_1, tensor_2):
        tensor_1_ = set(tensor_1.cpu().tolist())
        tensor_2_ = set(tensor_2.cpu().tolist())
        if len(tensor_1_.intersection(tensor_2_)) >= min_overlap:
            return True
        else:
            return False
    if loosely_equal(x[0],x[1]) and loosely_equal(x[1],x[2]) and loosely_equal(x[0],x[2]):
        return True
    else:
        return False

def analyze_span_lengths_and_count(text):
    mask_matches = list(re.finditer('<mask>', text))

    mask_info_dict= defaultdict(list)
    prev_mask = None
    span_count = 0
    curr_span_length = 1
    for i, mask in enumerate(mask_matches):
        
        if i == 0:
            mask_info_dict[span_count].append(i)
            
        else:
            if prev_mask.span()[1] == mask.span()[0]:
                mask_info_dict[span_count].append(i)
                curr_span_length += 1
            else:
                span_count += 1
                mask_info_dict[span_count].append(i)
                curr_span_length = 1
        prev_mask = mask

    span_lengths = []
    for span_id, span_len in mask_info_dict.items():
        
        span_lengths.append(len(span_len))
    return mask_info_dict, span_lengths

In [207]:

config = {}
config['max_tokens_per_span'] = 3
config['device'] = 'cuda'
config['k_per_location'] = 5
config['model_paths'] = ["gpt2-large", "/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-with-gpt2-large-embeds-energy-training/step_2800_best_checkpoint/"]
config['tokenizer_paths'] = config['model_paths']
config['losses'] = ["gpt2", "classification_no_prefix_logprobloss"]
config['model_types'] = ["AutoModelForCausalLM", "AutoModelForSequenceClassification"]
config['cache_dir'] = "/data/hyeryung/hf_cache"
config['build_loss_dict'] = json.loads('{"coeff_steps": 200, "coeff_pattern": "constant", "loss_type": "xentropy", "length_normalize": false, "AR_temperature": 1.0, "AR_top_k": 0, "AR_top_p": 0.96, "max_output_length": 20}')
config['target_type'] = 'embeds'
config['task'] = 'toxicity'

class dummyArgs:
        def __init__(self, **kwargs):
            for k, v in kwargs.items():
                setattr(self, k, v)

build_loss_args = dummyArgs(**config["build_loss_dict"])
build_loss_args.task = config["task"]

In [24]:

mlm = AutoModelForMaskedLM.from_pretrained('roberta-large').to(config['device'])
mlm_tokenizer = AutoTokenizer.from_pretrained('roberta-large')

In [154]:
intermediate_outputs = pd.read_json('outputs/toxicity/llm/bx3p1fwj/outputs_epsilon0.9.txt.intermediate', lines=True)
intermediate_outputs = intermediate_outputs.explode('generations').reset_index(drop=True)
intermediate_outputs_eda = intermediate_outputs.loc[intermediate_outputs['generations'].apply(len) != 0].reset_index(drop=True)
intermediate_outputs_eda['prompt'] = intermediate_outputs_eda['prompt'].apply(lambda x: x['text'])
intermediate_outputs_eda['masked_sentences'] = intermediate_outputs_eda['generations'].apply(lambda x: [item[1] for item in x.items() if 'mask' in item[0]])
intermediate_outputs_eda = intermediate_outputs_eda.explode('masked_sentences').reset_index(drop=True)

In [159]:
all_source_texts = intermediate_outputs_eda['prompt'].tolist()
all_masked_sentences = intermediate_outputs_eda['masked_sentences'].tolist()

In [6]:
mask_info_dicts = []
span_lengths_es = []
for test_sent in all_masked_sentences:
    
    mask_info_dict, span_lengths = analyze_span_lengths_and_count(test_sent)
    mask_info_dicts.append(mask_info_dict)
    span_lengths_es.append(span_lengths)

In [122]:
from collections import Counter
Counter([len(x) for x in span_lengths_es])

Counter({4: 379, 3: 186, 2: 59, 5: 277, 1: 4})

In [ ]:
## merge masks
source_texts = intermediate_outputs_eda['prompt'].tolist()
test_sent = all_masked_sentences[0]
test_sent_span_lengths = span_lengths_es[0]

special_token_ids = mlm_tokenizer.convert_tokens_to_ids(mlm_tokenizer.all_special_tokens)

In [208]:
comparison_results = []


for i in range(len(all_masked_sentences)):
    
    source_text = intermediate_outputs_eda['prompt'].tolist()[i]
    test_sent = all_masked_sentences[i]
    test_sent_span_lengths = span_lengths_es[i]
    test_sent_merged = re.sub(r"(<mask>)+", "<mask>", test_sent)
    # max_mask_cnt_per_span = [max(x, config['max_tokens_per_span']) for x in test_sent_span_lengths]
    mask_spans = [x.span() for x in re.finditer('<mask>',test_sent_merged)]
    
    base_hyp = test_sent_merged[:mask_spans[0][0]]
    curr_comparison_results = []
    for j in range(1, config['max_tokens_per_span']+1):
        curr_full_text_hyp = [base_hyp + "<mask>" * j + test_sent_merged[mask_spans[0][1]:]]
        # print(curr_full_text_hyp)
        # curr_queue_size = len(curr_full_text_hyp)
        
        inputs = mlm_tokenizer(
                    curr_full_text_hyp, return_tensors="pt", padding=True, truncation=True
        )
        inputs = inputs.to(config['device']) 
        masked_sequence=inputs['input_ids']
        
        prompt_enc=mlm_tokenizer(mlm_tokenizer.bos_token + source_text,add_special_tokens=False, return_tensors="pt", padding=True, truncation=True).to(config['device'])
        # prompt_enc['input_ids']=prompt_enc['input_ids'].expand(curr_queue_size,-1)
        # prompt_enc['attention_mask']=prompt_enc['attention_mask'].expand(curr_queue_size,-1)

        input_tokens = torch.cat([prompt_enc.input_ids, inputs.input_ids], dim=1).to(config['device'])
        attention_masks = torch.cat([prompt_enc.attention_mask, inputs.attention_mask], dim=1).to(config['device'])

        with torch.no_grad():
            logits = mlm(input_ids = input_tokens, 
                        attention_mask = attention_masks).logits
        
        logits[:, :, special_token_ids] = -float("inf")

        indices_in_mlm_tokens = (
            inputs.input_ids == mlm_tokenizer.mask_token_id
        ).nonzero(as_tuple=False) # if as_tuple=False, returns a tensor where column 1 indicates row indices, column 2 indicates column indices e.g. torch.Tensor([[0, 19],[0, 20], [0,38]])
        # print(f"-- location of masks including masks in the future spans: {indices_in_mlm_tokens}")

        # For each hypothesis in curr_full_text_hyp, first j mask locations are relevant
        # indices_in_mlm_tokens = torch.cat([x[:j] for x,j in zip(torch.chunk(indices_in_mlm_tokens, curr_queue_size), range(1, config['max_tokens_per_span']+1))],dim=0)
        indices_in_mlm_tokens = indices_in_mlm_tokens[:j]
        indices_in_mlm_tokens_0 = indices_in_mlm_tokens[:,0]
        indices_in_mlm_tokens_1 = indices_in_mlm_tokens[:,1]
        # print(f"-- location of masks (row indices): {indices_in_mlm_tokens_0}")
        # print(f"-- location of masks (col indices): {indices_in_mlm_tokens_1}")

        # Get top k tokens for the j masks
        predicted_token_ids = torch.topk(
            logits[indices_in_mlm_tokens_0, indices_in_mlm_tokens_1, :],
            k=config['k_per_location'],
            dim=-1,
        )            
        curr_comparison_results.append(predicted_token_ids.indices[0])

    comparison_results.append(curr_comparison_results)
    

In [213]:
equality_results = [check_equality(x) for x in comparison_results]
loose_equality_results = [check_loose_equality(x,4) for x in comparison_results]

In [182]:
sum(equality_results), len(equality_results)

(624, 905)

In [210]:
sum(equality_results), len(equality_results)

(309, 905)

In [204]:
sum(loose_equality_results), len(loose_equality_results)

(885, 905)

In [214]:
sum(loose_equality_results), len(loose_equality_results)

(869, 905)

In [ ]:
[x for x,y in zip(comparison_results,loose_equality_results) if y==False]

[[tensor([   6, 2156,    4,  209,  116], device='cuda:0'),
  tensor([   6, 2156,    4,  209,  116], device='cuda:0'),
  tensor([   6,    4, 2156,  113,   43], device='cuda:0')],
 [tensor([   9, 3243,   12, 1525, 1116], device='cuda:0'),
  tensor([   9,   12, 3243,   22,   19], device='cuda:0'),
  tensor([   9,   12, 3243,   22,   19], device='cuda:0')],
 [tensor([  323,  3720,  2234, 35271,  7737], device='cuda:0'),
  tensor([ 323, 3720, 2234, 3264, 7737], device='cuda:0'),
  tensor([  323,  3720,  2234, 35271,  4548], device='cuda:0')],
 [tensor([   8,   50, 4400,  359,   53], device='cuda:0'),
  tensor([   8,   50,  359, 4400,   53], device='cuda:0'),
  tensor([   8,   50,  359,  178, 4248], device='cuda:0')],
 [tensor([    6,  2156,   116,    60, 33647], device='cuda:0'),
  tensor([   6, 2156,   60,   12,  116], device='cuda:0'),
  tensor([    6,  2156,   131,    93, 33647], device='cuda:0')],
 [tensor([  279, 10736, 33780,   793,   397], device='cuda:0'),
  tensor([  279, 10736,   

## Prototype

In [1]:
import re
from collections import defaultdict
import json 
from typing import List, Tuple
from copy import deepcopy

from torch.utils.data import DataLoader,Dataset
import transformers
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import torch
import numpy as np
import pandas as pd
import wandb

from new_module.new_decode_utils import get_beam_hypotheses_v0
import new_module.losses as lossbuilder

In [2]:
# util 함수 선언

class CustomDataset(Dataset):
    def __init__(self, hypotheses_data:List[str]):
        self.hypotheses_data = hypotheses_data
        
    def __len__(self):
        return len(self.hypotheses_data)

    def __getitem__(self, idx:int):
        return self.hypotheses_data[idx]
    
    def __getitems__(self, idx:List[int]):
        return [self.hypotheses_data[j] for j in idx]
    
def repeat_interleave_unravel(arr,split_blocks):
    arr_ = torch.split(arr.T,1,dim=1)
    arr_ = [x.repeat(1,split_blocks[i]).reshape(-1,1) for i,x in enumerate(arr_)]
    arr_ = torch.cat(arr_,dim=0)
    return arr_

def analyze_span_lengths_and_count(text):
    mask_matches = list(re.finditer('<mask>', text))

    mask_info_dict= defaultdict(list)
    prev_mask = None
    span_count = 0
    curr_span_length = 1
    for i, mask in enumerate(mask_matches):
        
        if i == 0:
            mask_info_dict[span_count].append(i)
            
        else:
            if prev_mask.span()[1] == mask.span()[0]:
                mask_info_dict[span_count].append(i)
                curr_span_length += 1
            else:
                span_count += 1
                mask_info_dict[span_count].append(i)
                curr_span_length = 1
        prev_mask = mask

    span_lengths = []
    for span_id, span_len in mask_info_dict.items():
        
        span_lengths.append(len(span_len))
    return mask_info_dict, span_lengths

In [3]:
# 완성된 함수는 new_module/new_decode_utils_v2.py 에 저장되어 있음
from new_module.new_decode_utils_v2 import get_beam_hypotheses_v0_variable_length_v2, get_beam_hypotheses_v0_variable_length_considering_post_context_v2

### Prototype위한 변수 세팅

In [4]:
# prototype위한 변수 세팅
run_path = 'hayleyson/toxicity-decoding/bx3p1fwj'
api = wandb.Api()
run = api.run(run_path)
config = run.config
# config['model_paths'][0] = 'gpt2-large'
# config['tokenizer_paths'][0] = 'gpt2-large'

class dummyArgs:
        def __init__(self, **kwargs):
            for k, v in kwargs.items():
                setattr(self, k, v)

build_loss_args = dummyArgs(**config["build_loss_dict"])
build_loss_args.task = config["task"]

mlm = AutoModelForMaskedLM.from_pretrained('roberta-large').to(config['device'])
mlm_tokenizer = AutoTokenizer.from_pretrained('roberta-large')

## load tokenizer, models, define losses
name2tokenizer = {}
name2model = {}
name2config = {}
loss2tokenizer = {}
embed_luts = []

for i, model_path in enumerate(config["model_paths"]):
    if (
        model_path not in name2model
    ):  # making sure we are not loading the model twice in case some constraints use the same model.
        try:
            name2tokenizer[config["tokenizer_paths"][i]] = AutoTokenizer.from_pretrained(
                config["tokenizer_paths"][i],
                cache_dir=config["cache_dir"],
                use_fast=True,
            )
        except:
            name2tokenizer[config["tokenizer_paths"][i]] = AutoTokenizer.from_pretrained(
                config["tokenizer_paths"][i],
                cache_dir=config["cache_dir"],
                use_fast=False,
            )

        name2config[model_path] = AutoConfig.from_pretrained(
            model_path, cache_dir=config["cache_dir"]
        )

        if config["model_types"][i] == "RobertaCustomForSequenceClassification":
            pass
        else:
            name2model[model_path] = lossbuilder.ModelWrapper(
                getattr(transformers, config["model_types"][i]).from_pretrained(
                    model_path,
                    config=name2config[model_path],
                    cache_dir=config["cache_dir"],
                )
            )
        name2model[model_path].eval()
        name2model[model_path].to(config['device'])

lossfns = []
for i, loss in enumerate(config["losses"]):
    lossfns.append(
        lossbuilder.build_loss(
            loss,
            name2model[config["model_paths"][i]],
            name2tokenizer[config["tokenizer_paths"][i]],
            build_loss_args,
        )
    )
    lossfns[i].tokenizer.add_special_tokens({"mask_token": mlm_tokenizer.mask_token})
    loss2tokenizer[loss] = lossfns[i].tokenizer

special_token_ids = mlm_tokenizer.convert_tokens_to_ids(mlm_tokenizer.all_special_tokens)
intermediate_outputs = pd.read_json('outputs/toxicity/llm/bx3p1fwj/outputs_epsilon0.9.txt.intermediate', lines=True)
intermediate_outputs = intermediate_outputs.explode('generations').reset_index(drop=True)
intermediate_outputs_eda = intermediate_outputs.loc[intermediate_outputs['generations'].apply(len) != 0].reset_index(drop=True)
intermediate_outputs_eda['prompt'] = intermediate_outputs_eda['prompt'].apply(lambda x: x['text'])
intermediate_outputs_eda['masked_sentences'] = intermediate_outputs_eda['generations'].apply(lambda x: [item[1] for item in x.items() if 'mask' in item[0]])
intermediate_outputs_eda = intermediate_outputs_eda.explode('masked_sentences').reset_index(drop=True)
all_source_texts = intermediate_outputs_eda['prompt'].tolist()
all_masked_sentences = intermediate_outputs_eda['masked_sentences'].tolist()
mask_info_dicts = []
span_lengths_es = []
for test_sent in all_masked_sentences:
    
    mask_info_dict, span_lengths = analyze_span_lengths_and_count(test_sent)
    mask_info_dicts.append(mask_info_dict)
    span_lengths_es.append(span_lengths)

Starting new HTTPS connection (1): api.wandb.ai:443
https://api.wandb.ai:443 "POST /graphql HTTP/11" 200 None
https://api.wandb.ai:443 "POST /graphql HTTP/11" 200 None
Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /roberta-large/resolve/main/config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /roberta-large/resolve/main/tokenizer_config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/tokenizer_config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/config.json HTTP/11" 200 0


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/generation_config.json HTTP/11" 200 0


In [5]:
source_text = all_source_texts[0]
test_sent = all_masked_sentences[0]
test_sent_span_lengths = span_lengths_es[0]

In [6]:
# merge masks
test_sent_merged = re.sub(r"(<mask>)+", "<mask>", test_sent)

# Max number of mask tokens to replace each span
max_mask_cnt_per_span = [max(x, config['max_tokens_per_span']) for x in test_sent_span_lengths]

# Get the span information of merged masks in the test sentence
mask_spans = [x.span() for x in re.finditer('<mask>',test_sent_merged)]

queue = []
queue.append(test_sent_merged[:mask_spans[0][0]])
# for i in range(len(mask_spans)):
i = 0 ### for now, we only consider the first span
curr_queue_size = len(queue)

# candidate generation
curr_full_text_hyp = [base_hyp + "<mask>" * max_mask_cnt_per_span[i] + test_sent_merged[mask_spans[i][1]:] for base_hyp in queue]
## Tokenize & conduct MLM inference
inputs = mlm_tokenizer(
    curr_full_text_hyp, return_tensors="pt", padding=True, truncation=True
)
inputs = inputs.to(config['device']) 
masked_sequence=inputs['input_ids']

if config['consider_prompt_for_cand_gen']:
    
    prompt_enc=mlm_tokenizer(mlm_tokenizer.bos_token + source_text,add_special_tokens=False, return_tensors="pt", padding=True, truncation=True).to(config['device'])
    prompt_enc['input_ids']=prompt_enc['input_ids'].expand(curr_queue_size,-1)
    prompt_enc['attention_mask']=prompt_enc['attention_mask'].expand(curr_queue_size,-1)
    # print(f"-- shape of source text after being tokenized and converted to input ids: {prompt_enc['input_ids'].shape}")
    
    input_tokens = torch.cat([prompt_enc.input_ids, inputs.input_ids], dim=1).to(config['device'])
    attention_masks = torch.cat([prompt_enc.attention_mask, inputs.attention_mask], dim=1).to(config['device'])
    
    # with torch.no_grad():
    #     logits = mlm(**inputs).logits
    with torch.no_grad():
        logits = mlm(input_ids = input_tokens, 
                    attention_mask = attention_masks).logits

    # Choose top k among non-special tokens
    # print(f"-- shape of logits before removing source text part: {logits.shape}")
    logits = logits[:, prompt_enc.input_ids.shape[1]:]
    
else:
    with torch.no_grad():
        logits = mlm(**inputs).logits

## Choose top k among non-special tokens
logits[:, :, special_token_ids] = -float("inf")

indices_in_mlm_tokens = (
    inputs.input_ids == mlm_tokenizer.mask_token_id
).nonzero(as_tuple=False) # if as_tuple=False, returns a tensor where column 1 indicates row indices, column 2 indicates column indices e.g. torch.Tensor([[0, 19],[0, 20], [0,38]])
# print(f"-- location of masks including masks in the future spans: {indices_in_mlm_tokens}")

## get post context -- added 12/10
post_context = test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]]

## For each hypothesis in curr_full_text_hyp, first max_mask_cnt_per_span[i] mask locations are relevant
indices_in_mlm_tokens = torch.cat([x[:max_mask_cnt_per_span[i]] for x in torch.chunk(indices_in_mlm_tokens, curr_queue_size)],dim=0)
indices_in_mlm_tokens_0 = indices_in_mlm_tokens[:,0]
indices_in_mlm_tokens_1 = indices_in_mlm_tokens[:,1]
# print(f"-- location of masks (row indices): {indices_in_mlm_tokens_0}")
# print(f"-- location of masks (col indices): {indices_in_mlm_tokens_1}")

## Get top k tokens for the j masks
predicted_token_ids = torch.topk(
    logits[indices_in_mlm_tokens_0, indices_in_mlm_tokens_1, :],
    k=config['k_per_location'],
    dim=-1,
)            

## beam search에 넣기 전에 이런 작업을 해주는게 좋을까? ## right side를 아예 안볼거면 ok.  -> 꼭 해주지 않아도 같은 결과가 나오긴 함.
print(f"masked_sequence: {masked_sequence}")
masked_sequence = [masked_sequence[ix, :indices_in_mlm_tokens_1[max_mask_cnt_per_span[i]*(ix+1)-1]+1] for ix in range(masked_sequence.shape[0])]
print(f"masked_sequence: {masked_sequence}")
masked_sequence = torch.nn.utils.rnn.pad_sequence(masked_sequence, batch_first=True, padding_value=mlm_tokenizer.pad_token_id)
print(f"masked_sequence: {masked_sequence}")

masked_sequence: tensor([[    0,  3314,  6257,    11,   648,   277, 50264, 50264, 50264,    14,
          3639,     7,    49,  4935, 50264,   638,     4,   152,  1736,    18,
          6636, 50264,    13,     5,   488,     8, 16699,  1567,  4153,  3650,
          5586,    10,  8082,  6184,     9, 50264,    20,   665,  1427,     9,
         23616,   129,  4542,     7,   617, 22249,  4591,     5,   801,  4854,
            51,  7277,     7,  2313,     4,    85,    16,  4814, 15554,     7,
           192,   215, 26248, 19870,    13,     5,   659,     8,   157,    12,
          9442,     9,   643,     6,     8,    24,    16, 16553,    14,  3901,
          1797,    32,   551,     7,  1100,    42,  2256,  1856,  7444,    30,
             5,  1736,    11,   864,     4,     2]], device='cuda:0')
masked_sequence: [tensor([    0,  3314,  6257,    11,   648,   277, 50264, 50264, 50264],
       device='cuda:0')]
masked_sequence: tensor([[    0,  3314,  6257,    11,   648,   277, 50264, 50264, 50264

### Candidate generation을 max 개수 mask를 넣고 1번만 하고 그걸 이용해서 mask 개수 1,2,3개 경우에 대해서 한번에 디코딩


```python 
## 아래 같은 식으로 코드를 짜셔 쓰고 싶은데 여기에 들어갈 get_beam_hypotheses_v0_variable_length를 Prototype
hypotheses=list(queue) # deletion case
hypotheses.extend(get_beam_hypotheses_v0_variable_length(source_text, 
                        masked_sequence, 
                        (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
                        predicted_token_ids.indices,
                        mlm_tokenizer, 
                        lossfns,
                        config,
                        return_all_hypotheses=True)[0])
print(f"hypotheses: {hypotheses}")
if i < len(mask_spans) -1 :
    hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]] for x in hypotheses]
else:
    hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:] for x in hypotheses]

# Scoring the hypotheses and select top beam hypotheses
## 생략
top_beams = torch.topk(curr_loss, k=config['beam_size'], dim=-1, largest=False).indices
# print(f"-- selected {config['beam_size']} hypotheses' indices: {top_beams}")
# print(f"-- selected {config['beam_size']} hypotheses for current ({i}th) span: {[hypotheses_all[ix] for ix in top_beams]}")
queue = [hypotheses_all[ix] for ix in top_beams]
print(f"queue: {queue}")
```

In [165]:
# 인자 설정
source_text = source_text
masked_sequence = masked_sequence
indices_in_mlm_tokens = (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1)
predicted_token_ids = predicted_token_ids.indices
mlm_tokenizer = mlm_tokenizer
lossfns = lossfns
config = config

In [166]:
# 변수 초기화
final_hypotheses = [[] for i in range(len(masked_sequence))]
final_hypotheses_losses = [[] for i in range(len(masked_sequence))]
hypotheses = list(torch.split(masked_sequence,1,dim=0)) ## [torch.tensor([[a],[b],[c]]), torch.tensor([[d]])]
edit_indices = sorted(list(set(indices_in_mlm_tokens[1].tolist())))

print(f"hypotheses: {hypotheses}")
print(f"edit_indices: {edit_indices}")

hypotheses: [tensor([[    0,  3314,  6257,    11,   648,   277, 50264, 50264, 50264]],
       device='cuda:0')]
edit_indices: [6, 7, 8]


In [167]:
loss_weights = config['loss_weights']

# deletion 케이스 처리 : deletion case로 final_hypotheses, final_hypotheses_losses 초기화 
first_mask_token_indices = [indices_in_mlm_tokens[1][indices_in_mlm_tokens[0]==i][0] for i in range(len(hypotheses))]
final_hypotheses = [[hypotheses[i][:, :first_mask_token_indices[i]].squeeze(0)] for i in range(len(hypotheses))] # List[List[torch.tensor]] # 계속해서 누적될 hypotheses 목록
tmp_hypotheses = [x[0].tolist() for x in final_hypotheses] # List[List[int]] # 이번 j 에서 고려할 hypotheses
print(f"{tmp_hypotheses}")

curr_loss = torch.zeros(len(tmp_hypotheses)).to(config['device'])
for lossid, lossname in enumerate(config["losses"]):
    with torch.no_grad():
        lossvalue = lossfns[lossid].compute_gold_loss(
            source_text, mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True),
            label_id=config['target_label_ids'][lossid],
        )
    torch.cuda.empty_cache()
    curr_loss += loss_weights[lossid] * lossvalue
        
final_hypotheses_losses = [[x.squeeze(0)] for x in torch.split(curr_loss, 1)]
print(f"final_hypotheses_losses: {final_hypotheses_losses}")

[[0, 3314, 6257, 11, 648, 277]]
final_hypotheses_losses: [[tensor(2.838, device='cuda:0')]]


In [168]:
# for curr_edit_index in edit_indices:
curr_edit_index = edit_indices[0] ### no loop for now
    
batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

print(f"batch_ids_to_edit: {batch_ids_to_edit}")
print(f"num_initial_hypotheses: {num_initial_hypotheses}")
print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
print(f"tmp_hypotheses.shape: {tmp_hypotheses.shape}")
new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = new_func_candidates.to(config['device'])
print(f"new_func_candidates: {new_func_candidates}")
tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
print(f"tmp_hypotheses: {tmp_hypotheses}")
# loss_weights = [1 - config['closs_weight'], config['closs_weight']]
loss_weights = config['loss_weights']
curr_loss = torch.zeros(tmp_hypotheses.shape[0]).to(config['device'])
for lossid, lossname in enumerate(config["losses"]):
    with torch.no_grad():
        lossvalue = lossfns[lossid].compute_gold_loss(
            source_text, mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True),
            label_id=config['target_label_ids'][lossid],
        )
    torch.cuda.empty_cache()
    curr_loss += loss_weights[lossid] * lossvalue
print(f"curr_loss: {curr_loss}")
curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
print(f"curr_loss: {curr_loss}")
top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
print(f"top_beams: {top_beams}")
tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
print(f"tmp_hypotheses: {tmp_hypotheses}")
for jx, ix in enumerate(batch_ids_to_edit):
    hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
    final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
    final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
print("=== hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in hypotheses])
print("=== final_hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_hypotheses])
print("=== final_hypotheses_losses ===")
print(final_hypotheses_losses)

batch_ids_to_edit: [0]
num_initial_hypotheses: [1]
num_initial_tmp_hypotheses: [5]
tmp_hypotheses.shape: torch.Size([5, 9])
new_func_candidates: tensor([[1760, 1846, 4153, 1160,  403]], device='cuda:0')
new_func_candidates: tensor([[1760],
        [1846],
        [4153],
        [1160],
        [ 403]], device='cuda:0')
new_func_candidates: tensor([[1760],
        [1846],
        [4153],
        [1160],
        [ 403]], device='cuda:0')
tmp_hypotheses: tensor([[   0, 3314, 6257,   11,  648,  277, 1760],
        [   0, 3314, 6257,   11,  648,  277, 1846],
        [   0, 3314, 6257,   11,  648,  277, 4153],
        [   0, 3314, 6257,   11,  648,  277, 1160],
        [   0, 3314, 6257,   11,  648,  277,  403]], device='cuda:0')
curr_loss: tensor([3.217, 3.117, 3.505, 3.453, 3.456], device='cuda:0')
curr_loss: (tensor([3.217, 3.117, 3.505, 3.453, 3.456], device='cuda:0'),)
top_beams: [tensor([1, 0, 3], device='cuda:0')]
tmp_hypotheses: (tensor([[   0, 3314, 6257,   11,  648,  277, 1760],
 

In [134]:
curr_edit_index = edit_indices[1] ### 2nd mask 까지 decoding
    
batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

print(f"batch_ids_to_edit: {batch_ids_to_edit}")
print(f"num_initial_hypotheses: {num_initial_hypotheses}")
print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
print(f"tmp_hypotheses.shape: {tmp_hypotheses.shape}")
new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = new_func_candidates.to(config['device'])
print(f"new_func_candidates: {new_func_candidates}")
tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
print(f"tmp_hypotheses: {tmp_hypotheses}")
# loss_weights = [1 - config['closs_weight'], config['closs_weight']]
loss_weights = config['loss_weights']
curr_loss = torch.zeros(tmp_hypotheses.shape[0]).to(config['device'])
for lossid, lossname in enumerate(config["losses"]):
    with torch.no_grad():
        lossvalue = lossfns[lossid].compute_gold_loss(
            source_text, mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True),
            label_id=config['target_label_ids'][lossid],
        )
    torch.cuda.empty_cache()
    curr_loss += loss_weights[lossid] * lossvalue
print(f"curr_loss: {curr_loss}")
curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
print(f"curr_loss: {curr_loss}")
top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
print(f"top_beams: {top_beams}")
tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
print(f"tmp_hypotheses: {tmp_hypotheses}")
for jx, ix in enumerate(batch_ids_to_edit):
    hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
    final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
    final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
print("=== hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in hypotheses])
print("=== final_hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_hypotheses])
print("=== final_hypotheses_losses ===")
print(final_hypotheses_losses)

batch_ids_to_edit: [0]
num_initial_hypotheses: [3]
num_initial_tmp_hypotheses: [15]
tmp_hypotheses.shape: torch.Size([15, 9])
new_func_candidates: tensor([[   9, 1837,    6,   11, 4153]], device='cuda:0')
new_func_candidates: tensor([[   9],
        [   9],
        [   9],
        [1837],
        [1837],
        [1837],
        [   6],
        [   6],
        [   6],
        [  11],
        [  11],
        [  11],
        [4153],
        [4153],
        [4153]], device='cuda:0')
new_func_candidates: tensor([[   9],
        [   9],
        [   9],
        [1837],
        [1837],
        [1837],
        [   6],
        [   6],
        [   6],
        [  11],
        [  11],
        [  11],
        [4153],
        [4153],
        [4153]], device='cuda:0')
tmp_hypotheses: tensor([[   0, 3314, 6257,   11,  648,  277, 1846,    9],
        [   0, 3314, 6257,   11,  648,  277, 1760,    9],
        [   0, 3314, 6257,   11,  648,  277, 1160,    9],
        [   0, 3314, 6257,   11,  648,  277, 18

In [135]:
curr_edit_index = edit_indices[2] ### 2nd mask 까지 decoding
    
batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

print(f"batch_ids_to_edit: {batch_ids_to_edit}")
print(f"num_initial_hypotheses: {num_initial_hypotheses}")
print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
print(f"tmp_hypotheses.shape: {tmp_hypotheses.shape}")
new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = new_func_candidates.to(config['device'])
print(f"new_func_candidates: {new_func_candidates}")
tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
print(f"tmp_hypotheses: {tmp_hypotheses}")
# loss_weights = [1 - config['closs_weight'], config['closs_weight']]
loss_weights = config['loss_weights']
curr_loss = torch.zeros(tmp_hypotheses.shape[0]).to(config['device'])
for lossid, lossname in enumerate(config["losses"]):
    with torch.no_grad():
        lossvalue = lossfns[lossid].compute_gold_loss(
            source_text, mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True),
            label_id=config['target_label_ids'][lossid],
        )
    torch.cuda.empty_cache()
    curr_loss += loss_weights[lossid] * lossvalue
print(f"curr_loss: {curr_loss}")
curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
print(f"curr_loss: {curr_loss}")
top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
print(f"top_beams: {top_beams}")
tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
print(f"tmp_hypotheses: {tmp_hypotheses}")
for jx, ix in enumerate(batch_ids_to_edit):
    hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
    final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
    final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
print("=== hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in hypotheses])
print("=== final_hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_hypotheses])
print("=== final_hypotheses_losses ===")
print(final_hypotheses_losses)

batch_ids_to_edit: [0]
num_initial_hypotheses: [3]
num_initial_tmp_hypotheses: [15]
tmp_hypotheses.shape: torch.Size([15, 9])
new_func_candidates: tensor([[1846, 3650, 1760, 5751, 2883]], device='cuda:0')
new_func_candidates: tensor([[1846],
        [1846],
        [1846],
        [3650],
        [3650],
        [3650],
        [1760],
        [1760],
        [1760],
        [5751],
        [5751],
        [5751],
        [2883],
        [2883],
        [2883]], device='cuda:0')
new_func_candidates: tensor([[1846],
        [1846],
        [1846],
        [3650],
        [3650],
        [3650],
        [1760],
        [1760],
        [1760],
        [5751],
        [5751],
        [5751],
        [2883],
        [2883],
        [2883]], device='cuda:0')
tmp_hypotheses: tensor([[   0, 3314, 6257,   11,  648,  277, 1760,    9, 1846],
        [   0, 3314, 6257,   11,  648,  277, 1846,    6, 1846],
        [   0, 3314, 6257,   11,  648,  277, 1846,   11, 1846],
        [   0, 3314, 6257,   

In [138]:
final_top_beams = [torch.topk(torch.stack(x), k=config['beam_size'], dim=-1, largest=False).indices for x in final_hypotheses_losses]
final_final_hypotheses = [[x[i] for i in y] for x,y in zip(final_hypotheses,final_top_beams)]

In [139]:
print("=== final_final_hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_final_hypotheses])

=== final_final_hypotheses ===
[['<s>engaging in yet another', '<s>engaging in yet another crime', '<s>engaging in yet another act of']]


In [140]:
final_final_hypotheses_ = [torch.cat((x, masked_sequence[0, edit_indices[-1]+1:]), dim=-1) for x in final_final_hypotheses[0]]

In [141]:
print("=== final_final_hypotheses_ ===")
print([mlm_tokenizer.decode(x, skip_special_tokens=False) for x in final_final_hypotheses_])

=== final_final_hypotheses_ ===
['<s>engaging in yet another', '<s>engaging in yet another crime', '<s>engaging in yet another act of']


In [325]:
from copy import deepcopy
predicted_token_ids_backup = deepcopy(predicted_token_ids)

In [ ]:
#### 여태까지 해본 것을 for loop으로 묶기 

# 인자 설정
source_text = source_text
masked_sequence = masked_sequence
indices_in_mlm_tokens = (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1)
predicted_token_ids = predicted_token_ids.indices
mlm_tokenizer = mlm_tokenizer
lossfns = lossfns
config = config

# 변수 초기화
final_hypotheses = [[] for i in range(len(masked_sequence))]
final_hypotheses_losses = [[] for i in range(len(masked_sequence))]
hypotheses = list(torch.split(masked_sequence,1,dim=0)) ## [torch.tensor([[a],[b],[c]]), torch.tensor([[d]])]
edit_indices = sorted(list(set(indices_in_mlm_tokens[1].tolist())))

# print(f"hypotheses: {hypotheses}")
# print(f"edit_indices: {edit_indices}")

for curr_edit_index in edit_indices:
# curr_edit_index = edit_indices[0] ### no loop for now
        
    batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
    num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
    tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
    num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
    tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

    # print(f"batch_ids_to_edit: {batch_ids_to_edit}")
    # print(f"num_initial_hypotheses: {num_initial_hypotheses}")
    # print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
    # print(f"tmp_hypotheses.shape: {tmp_hypotheses.shape}")
    new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
    # print(f"new_func_candidates: {new_func_candidates}")
    new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
    # print(f"new_func_candidates: {new_func_candidates}")
    new_func_candidates = new_func_candidates.to(config['device'])
    # print(f"new_func_candidates: {new_func_candidates}")
    tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
    # print(f"tmp_hypotheses: {tmp_hypotheses}")
    # loss_weights = [1 - config['closs_weight'], config['closs_weight']]
    loss_weights = config['loss_weights']
    curr_loss = torch.zeros(tmp_hypotheses.shape[0]).to(config['device'])
    for lossid, lossname in enumerate(config["losses"]):
        with torch.no_grad():
            lossvalue = lossfns[lossid].compute_gold_loss(
                source_text, mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True),
                label_id=config['target_label_ids'][lossid],
            )
        torch.cuda.empty_cache()
        curr_loss += loss_weights[lossid] * lossvalue
    # print(f"curr_loss: {curr_loss}")
    curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
    # print(f"curr_loss: {curr_loss}")
    top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
    # print(f"top_beams: {top_beams}")
    tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
    # print(f"tmp_hypotheses: {tmp_hypotheses}")
    for jx, ix in enumerate(batch_ids_to_edit):
        hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
        final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
        final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
    # print("=== hypotheses ===")
    # print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in hypotheses])
    # print("=== final_hypotheses ===")
    # print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_hypotheses])
    # print("=== final_hypotheses_losses ===")
    # print(final_hypotheses_losses)

final_top_beams = [torch.topk(torch.stack(x), k=config['beam_size'], dim=-1, largest=False).indices for x in final_hypotheses_losses]
final_final_hypotheses = [[x[i] for i in y] for x,y in zip(final_hypotheses,final_top_beams)]

[mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_final_hypotheses]

hypotheses: [tensor([[    0,  3314,  6257,    11,   648,   277, 50264, 50264, 50264]],
       device='cuda:0')]
edit_indices: [6, 7, 8]
batch_ids_to_edit: [0]
num_initial_hypotheses: [1]
num_initial_tmp_hypotheses: [5]
tmp_hypotheses.shape: torch.Size([5, 9])
new_func_candidates: tensor([[1760, 1846, 4153, 1160,  403]], device='cuda:0')
new_func_candidates: tensor([[1760],
        [1846],
        [4153],
        [1160],
        [ 403]], device='cuda:0')
new_func_candidates: tensor([[1760],
        [1846],
        [4153],
        [1160],
        [ 403]], device='cuda:0')
tmp_hypotheses: tensor([[   0, 3314, 6257,   11,  648,  277, 1760],
        [   0, 3314, 6257,   11,  648,  277, 1846],
        [   0, 3314, 6257,   11,  648,  277, 4153],
        [   0, 3314, 6257,   11,  648,  277, 1160],
        [   0, 3314, 6257,   11,  648,  277,  403]], device='cuda:0')
curr_loss: tensor([3.217, 3.117, 3.505, 3.453, 3.456], device='cuda:0')
curr_loss: (tensor([3.217, 3.117, 3.505, 3.453, 3.456], d

In [327]:
print("=== final_final_hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_final_hypotheses])

=== final_final_hypotheses ===
[['<s>engaging in yet another crime', '<s>engaging in yet another act of', '<s>engaging in yet another act']]


##### 몇가지 변형 준 뒤 비교

In [ ]:
# (구현방안 1)
# get_beam_hypotheses_v0_variable_length_no_reranking 안에서 mask 개수 1,2,3에 대한 beam을 다 저장해서 리턴
# return val: List(str) 
# 함수 바깥에서 오른쪽 context 붙이고 reranking.
# 이렇게 하면 기존과 기능적으로 달라진게 없다. candidate generation 1번만 해서 발생하는 차이 빼고는..
# 지금 테스트하고 있는 케이스에서는 근데 위 implementation과 결과가 같다.

hypotheses=list(queue) # deletion case
hypotheses.extend(get_beam_hypotheses_v0_variable_length(source_text, 
                        masked_sequence, 
                        (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
                        predicted_token_ids.indices,
                        mlm_tokenizer, 
                        lossfns,
                        config,
                        return_all_hypotheses=True)[0])
print(f"hypotheses: {hypotheses}")
if i < len(mask_spans) -1 :
    hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]] for x in hypotheses]
else:
    hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:] for x in hypotheses]

# Scoring the hypotheses and select top beam hypotheses
loss_weights = config['loss_weights']
batch_size = 32

curr_loss = torch.zeros(len(hypotheses_all)).to(config['device'])
data_loader = DataLoader(CustomDataset(hypotheses_all),batch_size=batch_size)

for lossid, lossname in enumerate(config["losses"]):
    lossvalues=[]
    with torch.no_grad():
        for batch in data_loader:
            lossvalue = lossfns[lossid].compute_gold_loss(
                source_text, batch,
                label_id=config['target_label_ids'][lossid],
            )
            lossvalues.append(lossvalue)
            torch.cuda.empty_cache()
    lossvalue = torch.cat(lossvalues,dim=0)
    curr_loss += loss_weights[lossid] * lossvalue

torch.cuda.empty_cache()
top_beams = torch.topk(curr_loss, k=config['beam_size'], dim=-1, largest=False).indices
# print(f"-- selected {config['beam_size']} hypotheses' indices: {top_beams}")
# print(f"-- selected {config['beam_size']} hypotheses for current ({i}th) span: {[hypotheses_all[ix] for ix in top_beams]}")
queue = [hypotheses_all[ix] for ix in top_beams]
print(f"queue: {queue}")

hypotheses: ['engaging in yet another', 'engaging in yet another crime', 'engaging in yet another act', 'engaging in yet another incident', 'engaging in yet another act of', 'engaging in yet another crime,', 'engaging in yet another crime in', 'engaging in yet another act of theft', 'engaging in yet another act of crime', 'engaging in yet another crime, theft']
queue: ['engaging in yet another crime that adds to their extensive', 'engaging in yet another act that adds to their extensive', 'engaging in yet another act of theft that adds to their extensive']


In [ ]:
# (구현방안 2)
# get_beam_hypotheses_v0_variable_length 안에서 mask 개수 1,2,3에 대한 beam을 다 저장해서 그 중에 left-context만 보고 reranking해서 제일 좋은 beam개 return 
# return val: List(str) 
# 이렇게 했을 때 달라지는 점 -- 이전에는 뒤쪽 context까지 붙여서 final reranking 했는데 이제는 그냥 앞쪽까지만 보고 beam search 하게 됨.

hypotheses=list(queue) # deletion case
hypotheses.extend(get_beam_hypotheses_v0_variable_length(source_text, 
                        masked_sequence, 
                        (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
                        predicted_token_ids.indices,
                        mlm_tokenizer, 
                        lossfns,
                        config)[0])
print(f"hypotheses: {hypotheses}")
if i < len(mask_spans) -1 :
    hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]] for x in hypotheses]
else:
    hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:] for x in hypotheses]
queue = hypotheses_all
print(f"queue: {queue}")

hypotheses: ['engaging in yet another crime that adds to their extensive', 'engaging in yet another act that adds to their extensive', 'engaging in yet another act of theft that adds to their extensive', 'e', 'n', 'g', 'a', 'g', 'i', 'n', 'g', ' ', 'i', 'n', ' ', 'y', 'e', 't', ' ', 'a', 'n', 'o', 't', 'h', 'e', 'r', ' ', 'c', 'r', 'i', 'm', 'e']
queue: ['engaging in yet another crime that adds to their extensive that adds to their extensive', 'engaging in yet another act that adds to their extensive that adds to their extensive', 'engaging in yet another act of theft that adds to their extensive that adds to their extensive', 'e that adds to their extensive', 'n that adds to their extensive', 'g that adds to their extensive', 'a that adds to their extensive', 'g that adds to their extensive', 'i that adds to their extensive', 'n that adds to their extensive', 'g that adds to their extensive', '  that adds to their extensive', 'i that adds to their extensive', 'n that adds to their ext

In [ ]:
# (구현방안 2 _v2)
# get_beam_hypotheses_v0_variable_length 안에서 mask 개수 1,2,3에 대한 beam을 다 저장해서 그 중에 left-context만 보고 reranking해서 제일 좋은 beam개 return 
# return val: List(str) 
# 이렇게 했을 때 달라지는 점 -- 이전에는 뒤쪽 context까지 붙여서 final reranking 했는데 이제는 그냥 앞쪽까지만 보고 beam search 하게 됨.

hypotheses=[] # deletion case
hypotheses.extend(get_beam_hypotheses_v0_variable_length_v2(source_text, 
                        masked_sequence, 
                        (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
                        predicted_token_ids.indices,
                        mlm_tokenizer, 
                        lossfns,
                        config)[0][0])
print(f"hypotheses: {hypotheses}")
if i < len(mask_spans) -1 :
    hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]] for x in hypotheses]
else:
    hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:] for x in hypotheses]
queue = hypotheses_all
print(f"queue: {queue}")

hypotheses: ['engaging in yet another', 'engaging in yet another crime', 'engaging in yet another act of']
queue: ['engaging in yet another that adds to their extensive', 'engaging in yet another crime that adds to their extensive', 'engaging in yet another act of that adds to their extensive']


In [ ]:
# (구현방안 3)
# get_beam_hypotheses_v0_variable_length 안에서 mask 개수 1,2,3에 대한 beam을 다 저장해서 그 중에 post context까지 보고 reranking해서 제일 좋은 beam개 return 
# return val: List(str) 
# 이렇게 했을 때 달라지는 점 -- beam search 중간에 j=1,2.. 일 때 뽑히는 candidate이 달라질 수 있음

hypotheses=[x+post_context for x in list(queue)] # deletion case
hypotheses.extend(get_beam_hypotheses_v0_variable_length_considering_post_context(source_text, 
                        masked_sequence, 
                        post_context,
                        (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
                        predicted_token_ids.indices,
                        mlm_tokenizer, 
                        lossfns,
                        config)[0])
print(f"hypotheses: {hypotheses}")
queue = hypotheses
print(f"queue: {queue}")

hypotheses: ['engaging in yet another that adds to their extensive that adds to their extensive', 'engaging in yet another crime that adds to their extensive that adds to their extensive', 'engaging in yet another act of that adds to their extensive that adds to their extensive', 'e', 'n', 'g', 'a', 'g', 'i', 'n', 'g', ' ', 'i', 'n', ' ', 'y', 'e', 't', ' ', 'a', 'n', 'o', 't', 'h', 'e', 'r', ' ', 'c', 'r', 'i', 'm', 'e', ' ', 't', 'h', 'a', 't', ' ', 'a', 'd', 'd', 's', ' ', 't', 'o', ' ', 't', 'h', 'e', 'i', 'r', ' ', 'e', 'x', 't', 'e', 'n', 's', 'i', 'v', 'e']
queue: ['engaging in yet another that adds to their extensive that adds to their extensive', 'engaging in yet another crime that adds to their extensive that adds to their extensive', 'engaging in yet another act of that adds to their extensive that adds to their extensive', 'e', 'n', 'g', 'a', 'g', 'i', 'n', 'g', ' ', 'i', 'n', ' ', 'y', 'e', 't', ' ', 'a', 'n', 'o', 't', 'h', 'e', 'r', ' ', 'c', 'r', 'i', 'm', 'e', ' ', 't'

In [ ]:
# (구현방안 3 v2)
# get_beam_hypotheses_v0_variable_length 안에서 mask 개수 1,2,3에 대한 beam을 다 저장해서 그 중에 post context까지 보고 reranking해서 제일 좋은 beam개 return 
# return val: List(str) 
# 이렇게 했을 때 달라지는 점 -- beam search 중간에 j=1,2.. 일 때 뽑히는 candidate이 달라질 수 있음

hypotheses, scores = get_beam_hypotheses_v0_variable_length_considering_post_context_v2(source_text, 
                        masked_sequence, 
                        post_context,
                        (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
                        predicted_token_ids.indices,
                        mlm_tokenizer, 
                        lossfns,
                        config)
hypotheses = hypotheses[0]
print(f"hypotheses: {hypotheses}")
queue = hypotheses
print(f"queue: {queue}")

hypotheses: ['engaging in yet another crime that adds to their extensive', 'engaging in yet another act that adds to their extensive', 'engaging in yet another act of theft that adds to their extensive']
queue: ['engaging in yet another crime that adds to their extensive', 'engaging in yet another act that adds to their extensive', 'engaging in yet another act of theft that adds to their extensive']


In [7]:
# 각 방안 별로 결과가 어떻게 달라지는지 보다 여러 샘플에서 확인하기 
# --> /data/hyeryung/mucoco/new_module/new_decode_utils_v2.py 파일 및 /data/hyeryung/mucoco/new_module/sbatch_new_decode_utils_v2.sh 로 대체

In [ ]:
## 기존 코드
def editing_with_delete_variable_replace(source_text:str, test_sent:str, test_sent_span_lengths:List[int], 
                                         mlm:AutoModelForMaskedLM, mlm_tokenizer:AutoTokenizer, 
                                         lossfns:List[lossbuilder.BaseLoss], config: dict, batch_size:int=64) -> \
                                             Tuple[List[str],torch.FloatTensor,torch.BoolTensor,torch.FloatTensor]:
    
    """
    
    params: 
        source_text: a prompt text 
        test_sent: a masked text returned by LocateMachine     
        test_sent_span_lengths: a list of span lenghts for each mask span in the test_sent
        mlm:
        mlm_tokenizer:
        lossfns: 
        config:
        batch_size:             
    
    returns:
        hypotheses: list of one best hypothesis(editing result)
        best_weighted_loss: torch.FloatTensor of weighted loss for the best hypothesis.
        best_allsat: torch.ByteTensor of indicator(1,0) whether the best hypothesis satisfy cutoff (min_epsilons) for constraint energy score.
        best_logging_loss: torch.FloatTensor of shape (num samples, 2) of fluency energy score and constraint energy score for each best hypothesis.
    """
    
    # merge masks
    test_sent_merged = re.sub(r"(<mask>)+", "<mask>", test_sent)
    
    # Max number of mask tokens to replace each span
    max_mask_cnt_per_span = [max(x, config['max_tokens_per_span']) for x in test_sent_span_lengths]

    # Get the span information of merged masks in the test sentence
    mask_spans = [x.span() for x in re.finditer('<mask>',test_sent_merged)]

    special_token_ids = mlm_tokenizer.convert_tokens_to_ids(mlm_tokenizer.all_special_tokens)

    queue = []
    queue.append(test_sent_merged[:mask_spans[0][0]])
    for i in range(len(mask_spans)):
        curr_queue_size = len(queue)
        hypotheses=[list(queue)] # deletion case
        # print(f'working on the {i}th span. current queue size: {curr_queue_size}')
        
        for j in range(1, max_mask_cnt_per_span[i] + 1):
        # for j in range(1, 3):
            # print(f"   * appending {j} masks")
            # Set up sentence for MLM inference (note we append variable number mask and then the rest of the sentence where the other spans are masked)
            curr_full_text_hyp = [base_hyp + "<mask>" * j + test_sent_merged[mask_spans[i][1]:] for base_hyp in queue]
            # print(f"-- curr_masked_text: {curr_full_text_hyp}")
            
            # Tokenize & conduct MLM inference
            inputs = mlm_tokenizer(
                curr_full_text_hyp, return_tensors="pt", padding=True, truncation=True
            )
            inputs = inputs.to(config['device']) 
            masked_sequence=inputs['input_ids']
            
            if config['consider_prompt_for_cand_gen']:
            
                prompt_enc=mlm_tokenizer(mlm_tokenizer.bos_token + source_text,add_special_tokens=False, return_tensors="pt", padding=True, truncation=True).to(config['device'])
                prompt_enc['input_ids']=prompt_enc['input_ids'].expand(curr_queue_size,-1)
                prompt_enc['attention_mask']=prompt_enc['attention_mask'].expand(curr_queue_size,-1)
                # print(f"-- shape of source text after being tokenized and converted to input ids: {prompt_enc['input_ids'].shape}")
                
                input_tokens = torch.cat([prompt_enc.input_ids, inputs.input_ids], dim=1).to(config['device'])
                attention_masks = torch.cat([prompt_enc.attention_mask, inputs.attention_mask], dim=1).to(config['device'])
                
                # with torch.no_grad():
                #     logits = mlm(**inputs).logits
                with torch.no_grad():
                    logits = mlm(input_ids = input_tokens, 
                                attention_mask = attention_masks).logits

                # Choose top k among non-special tokens
                # print(f"-- shape of logits before removing source text part: {logits.shape}")
                logits = logits[:, prompt_enc.input_ids.shape[1]:]
                
            else:
                with torch.no_grad():
                    logits = mlm(**inputs).logits

            # Choose top k among non-special tokens
            logits[:, :, special_token_ids] = -float("inf")

            indices_in_mlm_tokens = (
                inputs.input_ids == mlm_tokenizer.mask_token_id
            ).nonzero(as_tuple=False) # if as_tuple=False, returns a tensor where column 1 indicates row indices, column 2 indicates column indices e.g. torch.Tensor([[0, 19],[0, 20], [0,38]])
            # print(f"-- location of masks including masks in the future spans: {indices_in_mlm_tokens}")
            
            # For each hypothesis in curr_full_text_hyp, first j mask locations are relevant
            indices_in_mlm_tokens = torch.cat([x[:j] for x in torch.chunk(indices_in_mlm_tokens, curr_queue_size)],dim=0)
            indices_in_mlm_tokens_0 = indices_in_mlm_tokens[:,0]
            indices_in_mlm_tokens_1 = indices_in_mlm_tokens[:,1]
            # print(f"-- location of masks (row indices): {indices_in_mlm_tokens_0}")
            # print(f"-- location of masks (col indices): {indices_in_mlm_tokens_1}")
            
            # Get top k tokens for the j masks
            predicted_token_ids = torch.topk(
                logits[indices_in_mlm_tokens_0, indices_in_mlm_tokens_1, :],
                k=config['k_per_location'],
                dim=-1,
            )            
            # print(f"-- top k tokens for each mask in each hypothesis: {predicted_token_ids.indices}")

            # When we do beam search, we only beam search up until the j masks.
            # print(f"-- shape of masked_sequence before slicing and padding: {masked_sequence.shape}")
            # print(f"-- masked_sequence before slicing and padding: {masked_sequence}")
            
            masked_sequence = [masked_sequence[ix, :indices_in_mlm_tokens_1[j*(ix+1)-1]+1] for ix in range(masked_sequence.shape[0])]
            masked_sequence = torch.nn.utils.rnn.pad_sequence(masked_sequence, batch_first=True, padding_value=mlm_tokenizer.pad_token_id)
            # print(f"-- shape of masked_sequence after slicing and padding: {masked_sequence.shape}")
            # print(f"-- masked_sequence after slicing and padding: {masked_sequence}")
            
            partial_hypotheses = get_beam_hypotheses_v0(source_text, 
                                masked_sequence, 
                                (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
                                predicted_token_ids.indices,
                                mlm_tokenizer, 
                                lossfns,
                                config)
            
            # print(f"-- num of returned partial hypotheses after beam search: {len(partial_hypotheses)}")
            # print(f"-- returned partial hypotheses after beam search: {partial_hypotheses}")
            
            partial_hypotheses = sum(partial_hypotheses, [])
            
            # print(f"-- num of partial hypotheses after unraveling disregarding the grouping by initial hypothesis: {len(partial_hypotheses)}")
            # print(f"-- partial hypotheses after unraveling disregarding the grouping by initial hypothesis: {partial_hypotheses}")
            # Extend the partial hypotheses to hypotheses pool for current mask span
            hypotheses.append(partial_hypotheses)

        # print("   * all mask length explored")
        hypotheses_all = sum(hypotheses, [])
        # print(f"-- # of hypotheses for current ({i}th) span: {len(hypotheses_all)}")
        # print(f"-- entire list of hypotheses for current ({i}th) span: {hypotheses_all}")
        
        # # Append snippet of text after current span and before next span before scoring
        if i < len(mask_spans) -1 :
            hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]] for x in hypotheses_all]
        else:
            hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:] for x in hypotheses_all]
        # print(f"-- entire list of hypotheses for current ({i}th) span for scoring: {hypotheses_all}") 
        
        # Scoring the hypotheses and select top beam hypotheses
        loss_weights = config['loss_weights']
        curr_loss = torch.zeros(len(hypotheses_all)).to(config['device'])
        data_loader = DataLoader(CustomDataset(hypotheses_all),batch_size=batch_size)

        for lossid, lossname in enumerate(config["losses"]):
            lossvalues=[]
            with torch.no_grad():
                for batch in data_loader:
                    lossvalue = lossfns[lossid].compute_gold_loss(
                        source_text, batch,
                        label_id=config['target_label_ids'][lossid],
                    )
                    lossvalues.append(lossvalue)
                    torch.cuda.empty_cache()
            lossvalue = torch.cat(lossvalues,dim=0)
            curr_loss += loss_weights[lossid] * lossvalue
        
        torch.cuda.empty_cache()
        top_beams = torch.topk(curr_loss, k=config['beam_size'], dim=-1, largest=False).indices
        # print(f"-- selected {config['beam_size']} hypotheses' indices: {top_beams}")
        # print(f"-- selected {config['beam_size']} hypotheses for current ({i}th) span: {[hypotheses_all[ix] for ix in top_beams]}")
        queue = [hypotheses_all[ix] for ix in top_beams]
        del hypotheses_all
        # print(f"-- queue after extending current step's selected hypotheses: {queue}")
        
    final_hypotheses, new_best_weighted_loss_, new_best_allsat_, new_best_logging_loss_ = final_reranking(source_text,
                                                                                                        [list(queue)],
                                                                                                        lossfns,
                                                                                                        config,
                                                                                                        batch_size=32)
    return final_hypotheses, new_best_weighted_loss_, new_best_allsat_, new_best_logging_loss_

In [ ]:
def editing_with_delete_variable_replace_cand_gen_once(source_text:str, test_sent:str, test_sent_span_lengths:List[int], 
                                         mlm:AutoModelForMaskedLM, mlm_tokenizer:AutoTokenizer, 
                                         lossfns:List[lossbuilder.BaseLoss], config: dict, batch_size:int=64) -> \
                                             Tuple[List[str],torch.FloatTensor,torch.BoolTensor,torch.FloatTensor]:
    
    
    # merge masks
    test_sent_merged = re.sub(r"(<mask>)+", "<mask>", test_sent)
    
    # Max number of mask tokens to replace each span
    max_mask_cnt_per_span = [max(x, config['max_tokens_per_span']) for x in test_sent_span_lengths]

    # Get the span information of merged masks in the test sentence
    mask_spans = [x.span() for x in re.finditer('<mask>',test_sent_merged)]

    special_token_ids = mlm_tokenizer.convert_tokens_to_ids(mlm_tokenizer.all_special_tokens)

    queue = []
    queue.append(test_sent_merged[:mask_spans[0][0]])
    for i in range(len(mask_spans)):
        curr_queue_size = len(queue)
        
        # candidate generation
        curr_full_text_hyp = [base_hyp + "<mask>" * max_mask_cnt_per_span[i] + test_sent_merged[mask_spans[i][1]:] for base_hyp in queue]
        ## Tokenize & conduct MLM inference
        inputs = mlm_tokenizer(
            curr_full_text_hyp, return_tensors="pt", padding=True, truncation=True
        )
        inputs = inputs.to(config['device']) 
        masked_sequence=inputs['input_ids']
        
        if config['consider_prompt_for_cand_gen']:
            
            prompt_enc=mlm_tokenizer(mlm_tokenizer.bos_token + source_text,add_special_tokens=False, return_tensors="pt", padding=True, truncation=True).to(config['device'])
            prompt_enc['input_ids']=prompt_enc['input_ids'].expand(curr_queue_size,-1)
            prompt_enc['attention_mask']=prompt_enc['attention_mask'].expand(curr_queue_size,-1)
            # print(f"-- shape of source text after being tokenized and converted to input ids: {prompt_enc['input_ids'].shape}")
            
            input_tokens = torch.cat([prompt_enc.input_ids, inputs.input_ids], dim=1).to(config['device'])
            attention_masks = torch.cat([prompt_enc.attention_mask, inputs.attention_mask], dim=1).to(config['device'])
            
            # with torch.no_grad():
            #     logits = mlm(**inputs).logits
            with torch.no_grad():
                logits = mlm(input_ids = input_tokens, 
                            attention_mask = attention_masks).logits

            # Choose top k among non-special tokens
            # print(f"-- shape of logits before removing source text part: {logits.shape}")
            logits = logits[:, prompt_enc.input_ids.shape[1]:]
            
        else:
            with torch.no_grad():
                logits = mlm(**inputs).logits

        ## Choose top k among non-special tokens
        logits[:, :, special_token_ids] = -float("inf")

        indices_in_mlm_tokens = (
            inputs.input_ids == mlm_tokenizer.mask_token_id
        ).nonzero(as_tuple=False) # if as_tuple=False, returns a tensor where column 1 indicates row indices, column 2 indicates column indices e.g. torch.Tensor([[0, 19],[0, 20], [0,38]])
        # print(f"-- location of masks including masks in the future spans: {indices_in_mlm_tokens}")
        
        ## For each hypothesis in curr_full_text_hyp, first max_mask_cnt_per_span[i] mask locations are relevant
        indices_in_mlm_tokens = torch.cat([x[:max_mask_cnt_per_span[i]] for x in torch.chunk(indices_in_mlm_tokens, curr_queue_size)],dim=0)
        indices_in_mlm_tokens_0 = indices_in_mlm_tokens[:,0]
        indices_in_mlm_tokens_1 = indices_in_mlm_tokens[:,1]
        # print(f"-- location of masks (row indices): {indices_in_mlm_tokens_0}")
        # print(f"-- location of masks (col indices): {indices_in_mlm_tokens_1}")
        
        ## Get top k tokens for the j masks
        predicted_token_ids = torch.topk(
            logits[indices_in_mlm_tokens_0, indices_in_mlm_tokens_1, :],
            k=config['k_per_location'],
            dim=-1,
        )            
        
        # beam search
        hypotheses=[list(queue)] # deletion case
        hypotheses+=get_beam_hypotheses_v0_variable_length(source_text, 
                                masked_sequence, 
                                (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
                                predicted_token_ids.indices,
                                mlm_tokenizer, 
                                lossfns,
                                config) 
        ## 뒤쪽 context까지 붙어있는 hypotheses들..
        queue = hypotheses
        # for j in range(1, max_mask_cnt_per_span[i] + 1):
        #     # print(f"   * appending {j} masks")
        #     # Set up sentence for MLM inference (note we append variable number mask and then the rest of the sentence where the other spans are masked)
        #     curr_full_text_hyp = [base_hyp + "<mask>" * j + test_sent_merged[mask_spans[i][1]:] for base_hyp in queue]
        #     # print(f"-- curr_masked_text: {curr_full_text_hyp}")
        
        #     ## Tokenize & conduct MLM inference
        #     inputs = mlm_tokenizer(
        #         curr_full_text_hyp, return_tensors="pt", padding=True, truncation=True
        #     )
        #     inputs = inputs.to(config['device']) 
        #     masked_sequence=inputs['input_ids']
                
        #     indices_in_mlm_tokens_0_j = torch.cat([x[:j] for x in torch.chunk(indices_in_mlm_tokens_0, curr_queue_size)],dim=0)
        #     indices_in_mlm_tokens_1_j = torch.cat([x[:j] for x in torch.chunk(indices_in_mlm_tokens_1, curr_queue_size)],dim=0)
        #     predicted_token_ids_j = torch.cat([x[:j] for x in torch.chunk(predicted_token_ids.indices, curr_queue_size)],dim=0)
        #     # predicted_token_ids.indices[]
            
        #     # When we do beam search, we only beam search up until the j masks.
        #     # print(f"-- shape of masked_sequence before slicing and padding: {masked_sequence.shape}")
        #     # print(f"-- masked_sequence before slicing and padding: {masked_sequence}")
            
        #     masked_sequence = [masked_sequence[ix, :indices_in_mlm_tokens_1_j[j*(ix+1)-1]+1] for ix in range(masked_sequence.shape[0])]
        #     masked_sequence = torch.nn.utils.rnn.pad_sequence(masked_sequence, batch_first=True, padding_value=mlm_tokenizer.pad_token_id)
        #     # print(f"-- shape of masked_sequence after slicing and padding: {masked_sequence.shape}")
        #     # print(f"-- masked_sequence after slicing and padding: {masked_sequence}")
            
        #     partial_hypotheses = get_beam_hypotheses_v0(source_text, 
        #                         masked_sequence, 
        #                         (indices_in_mlm_tokens_0_j, indices_in_mlm_tokens_1_j),
        #                         predicted_token_ids_j,
        #                         mlm_tokenizer, 
        #                         lossfns,
        #                         config)
            
        #     # print(f"-- num of returned partial hypotheses after beam search: {len(partial_hypotheses)}")
        #     # print(f"-- returned partial hypotheses after beam search: {partial_hypotheses}")
            
        #     partial_hypotheses = sum(partial_hypotheses, [])
            
        #     # print(f"-- num of partial hypotheses after unraveling disregarding the grouping by initial hypothesis: {len(partial_hypotheses)}")
        #     # print(f"-- partial hypotheses after unraveling disregarding the grouping by initial hypothesis: {partial_hypotheses}")
        #     # Extend the partial hypotheses to hypotheses pool for current mask span
        #     hypotheses.append(partial_hypotheses)

        # # print("   * all mask length explored")
        # hypotheses_all = sum(hypotheses, [])
        # # print(f"-- # of hypotheses for current ({i}th) span: {len(hypotheses_all)}")
        # # print(f"-- entire list of hypotheses for current ({i}th) span: {hypotheses_all}")
        
        # # # Append snippet of text after current span and before next span before scoring
        # if i < len(mask_spans) -1 :
        #     hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]] for x in hypotheses_all]
        # else:
        #     hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:] for x in hypotheses_all]
        # # print(f"-- entire list of hypotheses for current ({i}th) span for scoring: {hypotheses_all}") 
        
        # # Scoring the hypotheses and select top beam hypotheses
        # loss_weights = config['loss_weights']
        # curr_loss = torch.zeros(len(hypotheses_all)).to(config['device'])
        # data_loader = DataLoader(CustomDataset(hypotheses_all),batch_size=batch_size)

        # for lossid, lossname in enumerate(config["losses"]):
        #     lossvalues=[]
        #     with torch.no_grad():
        #         for batch in data_loader:
        #             lossvalue = lossfns[lossid].compute_gold_loss(
        #                 source_text, batch,
        #                 label_id=config['target_label_ids'][lossid],
        #             )
        #             lossvalues.append(lossvalue)
        #             torch.cuda.empty_cache()
        #     lossvalue = torch.cat(lossvalues,dim=0)
        #     curr_loss += loss_weights[lossid] * lossvalue
        
        # torch.cuda.empty_cache()
        # top_beams = torch.topk(curr_loss, k=config['beam_size'], dim=-1, largest=False).indices
        # # print(f"-- selected {config['beam_size']} hypotheses' indices: {top_beams}")
        # # print(f"-- selected {config['beam_size']} hypotheses for current ({i}th) span: {[hypotheses_all[ix] for ix in top_beams]}")
        # queue = [hypotheses_all[ix] for ix in top_beams]
        # del hypotheses_all
        # # print(f"-- queue after extending current step's selected hypotheses: {queue}")
        
    final_hypotheses, new_best_weighted_loss_, new_best_allsat_, new_best_logging_loss_ = final_reranking(source_text,
                                                                                                        [list(queue)],
                                                                                                        lossfns,
                                                                                                        config,
                                                                                                        batch_size=32)
    return final_hypotheses, new_best_weighted_loss_, new_best_allsat_, new_best_logging_loss_

### MLM을 안쓰고 CausalLM 으로만 candidate generation 하는 것 (확인사항 4)

In [27]:
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

In [7]:
source_text = all_source_texts[0]
test_sent = all_masked_sentences[0]
test_sent_span_lengths = span_lengths_es[0]

# merge masks
test_sent_merged = re.sub(r"(<mask>)+", "<mask>", test_sent)

# Max number of mask tokens to replace each span
max_mask_cnt_per_span = [max(x, config['max_tokens_per_span']) for x in test_sent_span_lengths]

# Get the span information of merged masks in the test sentence
mask_spans = [x.span() for x in re.finditer('<mask>',test_sent_merged)]

queue = []
queue.append(test_sent_merged[:mask_spans[0][0]])
# for i in range(len(mask_spans)):
i = 0 ### for now, we only consider the first span
curr_queue_size = len(queue)


In [ ]:

primary_model = AutoModelForCausalLM.from_pretrained('gpt2-large').to(config['device'])
primary_tokenizer = AutoTokenizer.from_pretrained('gpt2-large')
primary_tokenizer.add_special_tokens({'mask_token': mlm_tokenizer.mask_token,
                                      'pad_token': primary_tokenizer.eos_token})

https://huggingface.co:443 "HEAD /gpt2-large/resolve/main/config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /gpt2-large/resolve/main/generation_config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /gpt2-large/resolve/main/tokenizer_config.json HTTP/11" 200 0


2

In [10]:
source_batch = primary_tokenizer(source_text, return_tensors="pt").input_ids.to(config['device'])
predicted_batch = primary_tokenizer(test_sent_merged, return_tensors="pt").input_ids.to(config['device'])
beam_size = 3

indices_in_mlm_tokens = (
    predicted_batch == primary_tokenizer.mask_token_id
).nonzero(as_tuple=False) 

edit_token_index_primary = indices_in_mlm_tokens [: ,1]

In [60]:
hypotheses = torch.LongTensor([[]]).to(config['device'])
hyp_scores = torch.zeros(len(hypotheses), dtype = torch.float, device = config['device'])
seq_len = predicted_batch.size(-1)

for t in range(seq_len):
    
    prefix_added_hypotheses = torch.cat([source_batch.expand(hypotheses.size(0), -1), hypotheses], dim=-1)
    with torch.no_grad():
        model_output = primary_model(input_ids = prefix_added_hypotheses)

    logits_t = model_output.logits[:, -1, :] # get logits for the last timestep
    logp_t = F.log_softmax(logits_t, dim=-1) # (num_hypotheses, |V|)
    vocab_size = logits_t.size(-1)
    
    if t not in edit_token_index_primary:
        curr_nll = F.nll_loss(logp_t, predicted_batch[:, t].expand(logp_t.size(0)), reduction="none") # returns (num_hypotheses)
        hyp_scores = hyp_scores.expand_as(curr_nll) + curr_nll # (num_hypotheses)
        hypotheses = torch.cat([hypotheses, predicted_batch[:, t].expand(hypotheses.size(0), -1)], dim=-1)
    else:   
        contiuating_hyp_scores = (hyp_scores.unsqueeze(1).expand_as(logp_t) + (-logp_t)).view(-1) # (num_hypotheses x |V|)
        top_cand_hyp_scores, top_cand_hyp_pos = torch.topk(contiuating_hyp_scores, k=beam_size, largest=False)
        
        prev_hyp_ids = torch.div(top_cand_hyp_pos, vocab_size, rounding_mode='floor') # prev_hyp_id for each of top_cand_hyp. (beam_size)
        hyp_word_ids = top_cand_hyp_pos % vocab_size # hyp_word_id for each of top_cand_hyp. (beam_size)
        
        hypotheses = torch.cat([hypotheses[prev_hyp_ids], hyp_word_ids.unsqueeze(1)], dim=-1)
        hyp_scores = top_cand_hyp_scores

    torch.cuda.empty_cache()

In [ ]:
# LM으로만 candidate을 받아서 LM 으로 reranking 해서 beam search 하는 방법 (variable length)
hypotheses = torch.LongTensor([[]]).to(config['device'])
hypotheses_list = [torch.LongTensor([]).to(config['device'])]
hyp_scores = torch.zeros(len(hypotheses), dtype = torch.float, device = config['device'])
seq_len = predicted_batch.size(-1)
num_max_tokens = 3
primary_tokenizer.add_special_tokens({'pad_token': primary_tokenizer.eos_token})

for t in range(seq_len):
    
    if t not in edit_token_index_primary:
        prefix_added_hypotheses = torch.cat([source_batch.expand(hypotheses.size(0), -1), hypotheses], dim=-1)
        attention_masks = torch.where(prefix_added_hypotheses == primary_tokenizer.pad_token_id, 0, 1)
        with torch.no_grad():
            model_output = primary_model(input_ids = prefix_added_hypotheses,
                                         attention_mask = attention_masks)

        logits_t = model_output.logits[:, -1, :] # get logits for the last timestep
        logp_t = F.log_softmax(logits_t, dim=-1) # (num_hypotheses, |V|)
        vocab_size = logits_t.size(-1)
        
        curr_nll = F.nll_loss(logp_t, predicted_batch[:, t].expand(logp_t.size(0)), reduction="none") # returns (num_hypotheses)
        hyp_scores = hyp_scores.expand_as(curr_nll) + curr_nll # (num_hypotheses)
        # hypotheses = torch.cat([hypotheses, predicted_batch[:, t].expand(hypotheses.size(0), -1)], dim=-1)
        
        hypotheses_list = [torch.cat([hyp, predicted_batch[:, t]], dim=-1) for hyp in hypotheses_list]
        hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
        
    else:   
        total_hypotheses = []
        total_hyp_scores = []
        for j in range(num_max_tokens): 
            
            prefix_added_hypotheses = torch.cat([source_batch.expand(hypotheses.size(0), -1), hypotheses], dim=-1)
            attention_masks = torch.where(prefix_added_hypotheses == primary_tokenizer.pad_token_id, 0, 1)
            
            with torch.no_grad():
                model_output = primary_model(input_ids = prefix_added_hypotheses,
                                             attention_mask = attention_masks)

            logits_t = model_output.logits[:, -1, :] # get logits for the last timestep
            logp_t = F.log_softmax(logits_t, dim=-1) # (num_hypotheses, |V|)
            vocab_size = logits_t.size(-1)
            
            contiuating_hyp_scores = (hyp_scores.unsqueeze(1).expand_as(logp_t) + (-logp_t)).view(-1) # (num_hypotheses x |V|)
            top_cand_hyp_scores, top_cand_hyp_pos = torch.topk(contiuating_hyp_scores, k=beam_size, largest=False)
            
            prev_hyp_ids = torch.div(top_cand_hyp_pos, vocab_size, rounding_mode='floor') # prev_hyp_id for each of top_cand_hyp. (beam_size)
            hyp_word_ids = top_cand_hyp_pos % vocab_size # hyp_word_id for each of top_cand_hyp. (beam_dsize)
            
            # hypotheses = torch.cat([hypotheses[prev_hyp_ids], hyp_word_ids.unsqueeze(1)], dim=-1)
            hyp_scores = top_cand_hyp_scores
            
            prev_hyps = [hypotheses[idx][hypotheses[idx]!=primary_tokenizer.pad_token_id] for idx in prev_hyp_ids]
            hypotheses_list = [torch.cat([prev_hyps[i], hyp_word_ids[i].unsqueeze(0)], dim=-1) for i in range(beam_size)]
            hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
            
            total_hypotheses.extend(hypotheses_list)
            total_hyp_scores.append(hyp_scores)
        
        total_hyp_scores = torch.cat(total_hyp_scores, dim=0)
        top_beam_scores, top_beam_pos = torch.topk(total_hyp_scores, k=beam_size, largest=False)
        hypotheses_list = [total_hypotheses[ix] for ix in top_beam_pos]
        hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)

    torch.cuda.empty_cache()

In [51]:
from transformers import AutoModelForSequenceClassification
toxicity_model = AutoModelForSequenceClassification.from_pretrained('/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint')
toxicity_model = toxicity_model.to('cuda')

In [93]:
# LM으로만 candidate을 받아서 LM+EM 으로 reranking 해서 beam search 하는 방법 
# 다만 span 1개에 대해 length를 늘려가며 beam search 할 때는 em 으로 scoring 따로 안함
# LM+ EM reranking 시에 뒤에 오는 context를 고려 하지 않음
hypotheses = torch.LongTensor([[]]).to(config['device'])
hypotheses_list = [torch.LongTensor([]).to(config['device'])]
hyp_scores = torch.zeros(len(hypotheses), dtype = torch.float, device = config['device'])
seq_len = predicted_batch.size(-1)
num_max_tokens = 3
primary_tokenizer.add_special_tokens({'pad_token': primary_tokenizer.eos_token})

for t in range(seq_len):
    
    if t not in edit_token_index_primary:
        
        start1=time.time()
        prefix_added_hypotheses = torch.cat([source_batch.expand(hypotheses.size(0), -1), hypotheses], dim=-1)
        attention_masks = torch.where(prefix_added_hypotheses == primary_tokenizer.pad_token_id, 0, 1)
        with torch.no_grad():
            model_output = primary_model(input_ids = prefix_added_hypotheses,
                                         attention_mask = attention_masks)

        logits_t = model_output.logits[:, -1, :] # get logits for the last timestep
        logp_t = F.log_softmax(logits_t, dim=-1) # (num_hypotheses, |V|)
        vocab_size = logits_t.size(-1)
        
        curr_nll = F.nll_loss(logp_t, predicted_batch[:, t].expand(logp_t.size(0)), reduction="none") # returns (num_hypotheses)
        hyp_scores = hyp_scores.expand_as(curr_nll) + curr_nll # (num_hypotheses)
        # hypotheses = torch.cat([hypotheses, predicted_batch[:, t].expand(hypotheses.size(0), -1)], dim=-1)
        
        hypotheses_list = [torch.cat([hyp, predicted_batch[:, t]], dim=-1) for hyp in hypotheses_list]
        hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
        print(f"skip,{time.time()-start1}")
    else:   
        start2=time.time()
        total_hypotheses = []
        total_hyp_scores = []
        for j in range(num_max_tokens): 
            
            prefix_added_hypotheses = torch.cat([source_batch.expand(hypotheses.size(0), -1), hypotheses], dim=-1)
            attention_masks = torch.where(prefix_added_hypotheses == primary_tokenizer.pad_token_id, 0, 1)
            
            with torch.no_grad():
                model_output = primary_model(input_ids = prefix_added_hypotheses,
                                             attention_mask = attention_masks)

            logits_t = model_output.logits[:, -1, :] # get logits for the last timestep
            logp_t = F.log_softmax(logits_t, dim=-1) # (num_hypotheses, |V|)
            vocab_size = logits_t.size(-1)
            
            contiuating_hyp_scores = (hyp_scores.unsqueeze(1).expand_as(logp_t) + (-logp_t)).view(-1) # (num_hypotheses x |V|)
            top_cand_hyp_scores, top_cand_hyp_pos = torch.topk(contiuating_hyp_scores, k=beam_size, largest=False)
            
            prev_hyp_ids = torch.div(top_cand_hyp_pos, vocab_size, rounding_mode='floor') # prev_hyp_id for each of top_cand_hyp. (beam_size)
            hyp_word_ids = top_cand_hyp_pos % vocab_size # hyp_word_id for each of top_cand_hyp. (beam_dsize)
            
            # hypotheses = torch.cat([hypotheses[prev_hyp_ids], hyp_word_ids.unsqueeze(1)], dim=-1)
            hyp_scores = top_cand_hyp_scores
            
            prev_hyps = [hypotheses[idx][hypotheses[idx]!=primary_tokenizer.pad_token_id] for idx in prev_hyp_ids]
            hypotheses_list = [torch.cat([prev_hyps[i], hyp_word_ids[i].unsqueeze(0)], dim=-1) for i in range(beam_size)]
            hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
            
            total_hypotheses.extend(hypotheses_list)
            total_hyp_scores.append(hyp_scores)
        
        total_hyp_scores = torch.cat(total_hyp_scores, dim=0)
        total_hypotheses_ = torch.nn.utils.rnn.pad_sequence(total_hypotheses, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
        print(f"edit,{time.time()-start2}")
        
        start3=time.time()
        attention_masks_ = torch.where(total_hypotheses_ == primary_tokenizer.pad_token_id, 0, 1)
        with torch.no_grad():
            logits = toxicity_model(input_ids = total_hypotheses_,
                                    attention_mask = attention_masks_).logits
        logp_tox = F.log_softmax(logits, dim=-1)[:, 0]
        total_hyp_scores_ = total_hyp_scores + -10 * logp_tox
        
        top_beam_scores, top_beam_pos = torch.topk(total_hyp_scores_, k=beam_size, largest=False)
        hypotheses_list = [total_hypotheses[ix] for ix in top_beam_pos]
        hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
        hyp_scores = total_hyp_scores[top_beam_pos]
        print(f"rerank,{time.time()-start3}")

    torch.cuda.empty_cache()

skip,0.023571491241455078
skip,0.02184152603149414
skip,0.019522428512573242
skip,0.01998305320739746
skip,0.0197601318359375
edit,0.05882000923156738
rerank,0.008660078048706055
skip,0.018316030502319336
skip,0.01842045783996582
skip,0.019397735595703125
skip,0.018183469772338867
skip,0.018125295639038086
edit,0.05965590476989746
rerank,0.0077893733978271484
skip,0.021231651306152344
skip,0.02086329460144043
skip,0.020570039749145508
skip,0.02055215835571289
skip,0.021303892135620117
skip,0.019183635711669922
edit,0.0615382194519043
rerank,0.007494688034057617
skip,0.020676612854003906
skip,0.020646095275878906
skip,0.020888566970825195
skip,0.021030902862548828
skip,0.020296812057495117
skip,0.020982742309570312
skip,0.020850419998168945
skip,0.020934581756591797
skip,0.020984649658203125
skip,0.020874977111816406
skip,0.02082991600036621
skip,0.020837068557739258
skip,0.01996135711669922
edit,0.0684511661529541
rerank,0.007893085479736328
skip,0.020816326141357422
skip,0.02077269554

In [77]:
primary_tokenizer.batch_decode(hypotheses)

["engaging in yet another scam that adds to their extensive list of record. This individual's repeated attempts to ext for the law and tendency towards violent behavior highlights a concerning pattern of behavior The latest charge of blackmail only serves to further exemplify the potential danger they pose to society. It is deeply troubling to see such blatant disrespect for the rights and well-being of others, and it is imperative that appropriate measures are taken to address this ongoing threat posed by the individual in question.<|endoftext|>",
 "engaging in yet another scam that adds to their extensive list of record. This individual's repeated attempts to ext for the law and tendency towards violent behavior highlights a concerning pattern of criminal The latest charge of blackmail only serves to further exemplify the potential danger they pose to society. It is deeply troubling to see such blatant disrespect for the rights and well-being of others, and it is imperative that appr

In [ ]:
# 유사한 문제는 candidate generation을 MLM으로 해도, post context를 고려한 scoring이 없는 경우 발생 (method 2)
from new_module.new_decode_utils import get_beam_hypotheses_v0, get_beam_hypotheses_v1, get_combi_hypotheses, final_reranking, analyze_span_lengths_and_count, editing_with_delete_variable_replace

# merge masks
method="2"
test_sent_merged = re.sub(r"(<mask>)+", "<mask>", test_sent)

# Max number of mask tokens to replace each span
max_mask_cnt_per_span = [max(x, config['max_tokens_per_span']) for x in test_sent_span_lengths]

# Get the span information of merged masks in the test sentence
mask_spans = [x.span() for x in re.finditer('<mask>',test_sent_merged)]

queue = []
queue.append(test_sent_merged[:mask_spans[0][0]])
for i in range(len(mask_spans)):
    curr_queue_size = len(queue)

    # candidate generation
    curr_full_text_hyp = [base_hyp + "<mask>" * max_mask_cnt_per_span[i] + test_sent_merged[mask_spans[i][1]:] for base_hyp in queue]
    ## Tokenize & conduct MLM inference
    inputs = mlm_tokenizer(
        curr_full_text_hyp, return_tensors="pt", padding=True, truncation=True
    )
    inputs = inputs.to(config['device']) 
    masked_sequence=inputs['input_ids']

    if config['consider_prompt_for_cand_gen']:
        
        prompt_enc=mlm_tokenizer(mlm_tokenizer.bos_token + source_text,add_special_tokens=False, return_tensors="pt", padding=True, truncation=True).to(config['device'])
        prompt_enc['input_ids']=prompt_enc['input_ids'].expand(curr_queue_size,-1)
        prompt_enc['attention_mask']=prompt_enc['attention_mask'].expand(curr_queue_size,-1)
        
        input_tokens = torch.cat([prompt_enc.input_ids, inputs.input_ids], dim=1).to(config['device'])
        attention_masks = torch.cat([prompt_enc.attention_mask, inputs.attention_mask], dim=1).to(config['device'])
        
        with torch.no_grad():
            logits = mlm(input_ids = input_tokens, 
                        attention_mask = attention_masks).logits

        # Choose top k among non-special tokens
        logits = logits[:, prompt_enc.input_ids.shape[1]:]
        
    else:
        with torch.no_grad():
            logits = mlm(**inputs).logits

    ## Choose top k among non-special tokens
    logits[:, :, special_token_ids] = -float("inf")

    indices_in_mlm_tokens = (
        inputs.input_ids == mlm_tokenizer.mask_token_id
    ).nonzero(as_tuple=False) # if as_tuple=False, returns a tensor where column 1 indicates row indices, column 2 indicates column indices e.g. torch.Tensor([[0, 19],[0, 20], [0,38]])
    
    ## get post context 
    if i == len(mask_spans) -1:
        post_context = test_sent_merged[mask_spans[i][1]:]
    else:
        post_context = test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]]

    ## For each hypothesis in curr_full_text_hyp, first max_mask_cnt_per_span[i] mask locations are relevant
    indices_in_mlm_tokens = torch.cat([x[:max_mask_cnt_per_span[i]] for x in torch.chunk(indices_in_mlm_tokens, curr_queue_size)],dim=0)
    indices_in_mlm_tokens_0 = indices_in_mlm_tokens[:,0]
    indices_in_mlm_tokens_1 = indices_in_mlm_tokens[:,1]

    ## Get top k tokens for the j masks
    predicted_token_ids = torch.topk(
        logits[indices_in_mlm_tokens_0, indices_in_mlm_tokens_1, :],
        k=config['k_per_location'],
        dim=-1,
    )            

    ## beam search에 넣기 전에 이런 작업을 해주는게 좋을까? ## right side를 아예 안볼거면 ok.  -> 꼭 해주지 않아도 indices_in_mlm_tokens 에서 현재 span까지만 index를 뽑기 때문에 같은 결과가 나오긴 함.
    masked_sequence = [masked_sequence[ix, :indices_in_mlm_tokens_1[max_mask_cnt_per_span[i]*(ix+1)-1]+1] for ix in range(masked_sequence.shape[0])]
    masked_sequence = torch.nn.utils.rnn.pad_sequence(masked_sequence, batch_first=True, padding_value=mlm_tokenizer.pad_token_id)        
    
    if method == "2":
        
        # hypotheses=list(queue) # deletion case
        # hypotheses.extend(get_beam_hypotheses_v0_variable_length(source_text, 
        hypotheses, scores = get_beam_hypotheses_v0_variable_length_v2(source_text, # deletion 케이스도 beam 안에서 고려
                                masked_sequence, 
                                (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
                                predicted_token_ids.indices,
                                mlm_tokenizer, 
                                lossfns,
                                config)
        hypotheses = hypotheses[0]
        # print(f"hypotheses: {hypotheses}")
        if i < len(mask_spans) -1 :
            hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]] for x in hypotheses]
        else:
            hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:] for x in hypotheses]
        
        if i == len(mask_spans) -1:
            queue, new_best_weighted_loss_, new_best_allsat_, new_best_logging_loss_ = final_reranking(source_text,
                                                                                                        [hypotheses_all],
                                                                                                        lossfns,
                                                                                                        config,
                                                                                                        batch_size=32)
        else:
            queue = hypotheses_all
        # print(f"queue: {queue}")
        # for item in queue:
        #     print(item)
        # print('')
        
        
    torch.cuda.empty_cache()

In [66]:
hypotheses_all

["engaging in yet another that adds to their extensive record. This individual's repeated for the law and tendency towards violent behavior highlights a concerning pattern of The latest charge of blackmail only serves to further exemplify the potential danger they pose to society. It is deeply troubling to see such blatant disrespect for the rights and well-being of others, and it is imperative that appropriate measures are taken to address this ongoing threat posed by the individual in question.",
 "engaging in yet another that adds to their extensive record. This individual's repeated for the law and tendency towards violent behavior highlights a concerning pattern of criminal The latest charge of blackmail only serves to further exemplify the potential danger they pose to society. It is deeply troubling to see such blatant disrespect for the rights and well-being of others, and it is imperative that appropriate measures are taken to address this ongoing threat posed by the individua

In [81]:
edit_token_index_primary

tensor([ 5, 11, 18, 32], device='cuda:0')

In [91]:
import time

In [106]:
start0=time.time()
# LM으로만 candidate을 받아서 LM+EM 으로 reranking 해서 beam search 하는 방법 
# 다만 span 1개에 대해 length를 늘려가며 beam search 할 때는 em 으로 scoring 따로 안함
# LM+ EM reranking 시에 뒤에 오는 context를 고려 하지 않음
hypotheses = torch.LongTensor([[]]).to(config['device'])
hypotheses_list = [torch.LongTensor([]).to(config['device'])]
hyp_scores = torch.zeros(len(hypotheses), dtype = torch.float, device = config['device'])
seq_len = predicted_batch.size(-1)
num_max_tokens = 3
primary_tokenizer.add_special_tokens({'pad_token': primary_tokenizer.eos_token})

for t in range(seq_len):
    # print(t)
    
    if t not in edit_token_index_primary:
        start1=time.time()
        # print(f"hypotheses.size: {hypotheses.size()}")
        prefix_added_hypotheses = torch.cat([source_batch.expand(hypotheses.size(0), -1), hypotheses], dim=-1)
        attention_masks = torch.where(prefix_added_hypotheses == primary_tokenizer.pad_token_id, 0, 1)
        with torch.no_grad():
            model_output = primary_model(input_ids = prefix_added_hypotheses,
                                         attention_mask = attention_masks)

        logits_t = model_output.logits[:, -1, :] # get logits for the last timestep
        logp_t = F.log_softmax(logits_t, dim=-1) # (num_hypotheses, |V|)
        vocab_size = logits_t.size(-1)
        
        curr_nll = F.nll_loss(logp_t, predicted_batch[:, t].expand(logp_t.size(0)), reduction="none") # returns (num_hypotheses)
        hyp_scores = hyp_scores.expand_as(curr_nll) + curr_nll # (num_hypotheses)
        # hypotheses = torch.cat([hypotheses, predicted_batch[:, t].expand(hypotheses.size(0), -1)], dim=-1)
        
        hypotheses_list = [torch.cat([hyp, predicted_batch[:, t]], dim=-1) for hyp in hypotheses_list]
        hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
        # print(f"skip,{time.time()-start1}")
    else:   
        start2=time.time()
        if hypotheses.size(0) > beam_size:
            print(primary_tokenizer.batch_decode(hypotheses))
            # start2=time.time()
            # print(f"hypotheses.shape: {hypotheses.shape}")
            ## rerank with post context
            attention_masks_ = torch.where(hypotheses == primary_tokenizer.pad_token_id, 0, 1)
            with torch.no_grad():
                logits = toxicity_model(input_ids = hypotheses,
                                        attention_mask = attention_masks_).logits
            logp_tox = F.log_softmax(logits, dim=-1)[:, 0]
            hyp_scores_ = hyp_scores + -10 * logp_tox
            
            top_beam_scores, top_beam_pos = torch.topk(hyp_scores_, k=beam_size, largest=False)
            hypotheses_list = [hypotheses[ix] for ix in top_beam_pos]
            hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
            hyp_scores = hyp_scores[top_beam_pos]
            print('**')
            print(primary_tokenizer.batch_decode(hypotheses))
            print('--')
            print(top_beam_pos)
            print(torch.topk(hyp_scores, k=beam_size, largest=False)[1])
            # print(f"rerank,{time.time()-start2}")
        
        start3=time.time()
        total_hypotheses = []
        total_hyp_scores = []
        for j in range(num_max_tokens): 
            
            prefix_added_hypotheses = torch.cat([source_batch.expand(hypotheses.size(0), -1), hypotheses], dim=-1)
            attention_masks = torch.where(prefix_added_hypotheses == primary_tokenizer.pad_token_id, 0, 1)
            
            with torch.no_grad():
                model_output = primary_model(input_ids = prefix_added_hypotheses,
                                             attention_mask = attention_masks)

            logits_t = model_output.logits[:, -1, :] # get logits for the last timestep
            logp_t = F.log_softmax(logits_t, dim=-1) # (num_hypotheses, |V|)
            vocab_size = logits_t.size(-1)
            
            contiuating_hyp_scores = (hyp_scores.unsqueeze(1).expand_as(logp_t) + (-logp_t)).view(-1) # (num_hypotheses x |V|)
            top_cand_hyp_scores, top_cand_hyp_pos = torch.topk(contiuating_hyp_scores, k=beam_size, largest=False)
            
            prev_hyp_ids = torch.div(top_cand_hyp_pos, vocab_size, rounding_mode='floor') # prev_hyp_id for each of top_cand_hyp. (beam_size)
            hyp_word_ids = top_cand_hyp_pos % vocab_size # hyp_word_id for each of top_cand_hyp. (beam_dsize)
            
            # hypotheses = torch.cat([hypotheses[prev_hyp_ids], hyp_word_ids.unsqueeze(1)], dim=-1)
            hyp_scores = top_cand_hyp_scores
            
            prev_hyps = [hypotheses[idx][hypotheses[idx]!=primary_tokenizer.pad_token_id] for idx in prev_hyp_ids]
            hypotheses_list = [torch.cat([prev_hyps[i], hyp_word_ids[i].unsqueeze(0)], dim=-1) for i in range(beam_size)]
            hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
            
            total_hypotheses.extend(hypotheses_list)
            total_hyp_scores.append(hyp_scores)
        
        hypotheses_list = total_hypotheses
        hypotheses = torch.nn.utils.rnn.pad_sequence(total_hypotheses, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
        hyp_scores = torch.cat(total_hyp_scores, dim=0)
        
        # print(f"edit,{time.time()-start3}")
        # total_hyp_scores = torch.cat(total_hyp_scores, dim=0)
        # total_hypotheses_ = torch.nn.utils.rnn.pad_sequence(total_hypotheses, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
        # attention_masks_ = torch.where(prefix_added_hypotheses == primary_tokenizer.pad_token_id, 0, 1)
        # with torch.no_grad():
        #     logits = toxicity_model(input_ids = total_hypotheses_,
        #                             attention_mask = attention_masks_).logits
        # logp_tox = F.log_softmax(logits, dim=-1)[:, 0]
        # total_hyp_scores_ = total_hyp_scores + -10 * logp_tox
        
        # top_beam_scores, top_beam_pos = torch.topk(total_hyp_scores_, k=beam_size, largest=False)
        # hypotheses_list = [total_hypotheses[ix] for ix in top_beam_pos]
        # hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
        # hyp_scores = total_hyp_scores[top_beam_pos]

    torch.cuda.empty_cache()


attention_masks_ = torch.where(hypotheses == primary_tokenizer.pad_token_id, 0, 1)
with torch.no_grad():
    logits = toxicity_model(input_ids = hypotheses,
                            attention_mask = attention_masks_).logits
logp_tox = F.log_softmax(logits, dim=-1)[:, 0]
hyp_scores_ = hyp_scores + -10 * logp_tox

top_beam_scores, top_beam_pos = torch.topk(hyp_scores_, k=beam_size, largest=False)
hypotheses_list = [hypotheses[ix] for ix in top_beam_pos]
hypotheses = torch.nn.utils.rnn.pad_sequence(hypotheses_list, batch_first=True, padding_value=primary_tokenizer.pad_token_id)
hyp_scores = hyp_scores[top_beam_pos]
# print(time.time()-start0)

['engaging in yet another extortion that adds to their extensive<|endoftext|><|endoftext|>', 'engaging in yet another scam that adds to their extensive<|endoftext|><|endoftext|>', 'engaging in yet another sex that adds to their extensive<|endoftext|><|endoftext|>', 'engaging in yet another extortion scheme that adds to their extensive<|endoftext|>', 'engaging in yet another scam involving that adds to their extensive<|endoftext|>', 'engaging in yet another extortion scam that adds to their extensive<|endoftext|>', 'engaging in yet another extortion scheme. that adds to their extensive', 'engaging in yet another extortion scam. that adds to their extensive', 'engaging in yet another extortion scheme involving that adds to their extensive']
**
['engaging in yet another extortion scheme. that adds to their extensive', 'engaging in yet another extortion scam. that adds to their extensive', 'engaging in yet another extortion scheme involving that adds to their extensive']
--
tensor([6, 7, 8

In [103]:
primary_tokenizer.batch_decode(hypotheses)

["engaging in yet another extortion scheme. that adds to their extensive list of crimes record. This individual's repeated attempts to ext for the law and tendency towards violent behavior highlights a concerning pattern of criminal behavior. The latest charge of blackmail only serves to further exemplify the potential danger they pose to society. It is deeply troubling to see such blatant disrespect for the rights and well-being of others, and it is imperative that appropriate measures are taken to address this ongoing threat posed by the individual in question.",
 "engaging in yet another extortion scheme. that adds to their extensive list of crimes record. This individual's repeated attempts to ext for the law and tendency towards violent behavior highlights a concerning pattern of behavior.\n The latest charge of blackmail only serves to further exemplify the potential danger they pose to society. It is deeply troubling to see such blatant disrespect for the rights and well-being o

## Evaluating after running different versions using new_decode_utils_v2.py

In [2]:
import joblib
import pandas as pd
import random 
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from langdetect import detect
from evaluation.prompted_sampling.evaluate import (
    conditional_perplexity,
    distinctness,
    fluency_classify,
    formality_score_ext,
    formality_score_int,
    repetition,
    sentiment_classify_big,
    sentiment_classify_own2,
    toxicity_score,
    toxicity_score_energy,
    toxicity_score_int,
    toxicity_score_mucola,
    nli_score,
    sentiment_classify_gpt4o,
    contents_preservation_metrics,
    save_qualitative_results
)

def is_english(text):
    try:
        return detect(text) == 'en'  # 'en'은 영어를 의미
    except:
        return False  # 예외 발생 시 False 처리
    
device = 'cuda'
# torch.cuda.empty_cache()
# eval_model = AutoModelForCausalLM.from_pretrained('gpt2-xl').to('cuda')
# eval_tokenizer = AutoTokenizer.from_pretrained('gpt2-xl')
# torch.cuda.empty_cache()

# intermediate_outputs = pd.read_json('outputs/toxicity/llm/bx3p1fwj/outputs_epsilon0.9.txt.intermediate', lines=True)
# intermediate_outputs = intermediate_outputs.explode('generations').reset_index(drop=True)
# intermediate_outputs_eda = intermediate_outputs.loc[intermediate_outputs['generations'].apply(len) != 0].reset_index(drop=True)
# intermediate_outputs_eda['prompt'] = intermediate_outputs_eda['prompt'].apply(lambda x: x['text'])
# print(f"Number of total samples: {len(intermediate_outputs_eda)}")
# all_source_texts = intermediate_outputs_eda['prompt'].tolist()

# random.seed(999)
# idxes_for_test = random.sample(range(len(all_source_texts)),50)
# print(f"Number of samples: {len(idxes_for_test)}")

# prompts = [all_source_texts[i] for i in idxes_for_test]


with open('/data/hyeryung/mucoco/new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150_below_nontoxic_threshold_0_95_332_index.txt', 'r') as f:
    indices = f.read().split()
indices = [int(x) for x in indices]
located_data = pd.read_json('/data/hyeryung/mucoco/new_module/locate/locate_num_tokens_eda/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150_nontoxic_locate_max_7.jsonl', lines=True)
located_data = located_data.explode('generations').reset_index(drop=True)
located_data = located_data.loc[located_data['generations'].apply(len) != 0].reset_index(drop=True)

located_data['prompt'] = located_data['prompt'].apply(lambda x: x['text'])
located_data['masked_sentences'] = located_data['generations'].apply(lambda x: x['text'])
located_data = located_data.loc[indices,:].copy()
print(f"Number of total samples: {len(located_data)}")
all_source_texts = located_data['prompt'].tolist()
all_masked_sentences = located_data['masked_sentences'].tolist()

num_test_samples = 50
random.seed(999)
idxes_for_test = random.sample(range(len(all_source_texts)),num_test_samples)
prompts = [all_source_texts[i] for i in idxes_for_test]

Number of total samples: 332


In [4]:
device = 'cuda'
torch.cuda.empty_cache()
eval_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-14B', torch_dtype = torch.float16).to('cuda')
eval_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-14B')
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [6]:
# load generation file
results = joblib.load('/data/hyeryung/mucoco/new_module/decoding_result_using_v1_gpt2-large.pkl')
results = sum(results, [])

generations_df = pd.DataFrame({
    'prompt': [{'text': x} for x in prompts],
    'generations': [[{'text': x}] for x in results]
})

In [7]:

generations_df2 = generations_df
with torch.no_grad():
    ppl, total_ppl = conditional_perplexity(generations_df2, eval_model, eval_tokenizer, device=device, write_file=None)
print(ppl, total_ppl)
toxicity_model_path = '/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint/'
toxicity_model_type = 'AutoModelForSequenceClassification'
(avg_max_toxicity, toxic_probability_p, avg_toxicity, toxic_probability_s) = toxicity_score_int(generations_df, None, device,
                                                                                             toxicity_model_path, toxicity_model_type)
print(avg_toxicity, toxic_probability_s)
fluency = fluency_classify(generations_df2, None)
print(fluency)

Evaluating PPL: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  9.33it/s]


6.566638219506582 50 50 50
[7.952477116403141, 7.529837439058022, 7.737002269743101, 3.6963434676385516, 4.8750420860558625, 4.955344683911249, 5.536395438058603, 6.789924648806377, 10.13227187375179, 9.316419520411483, 8.047893309692856, 5.880175783227683, 5.089969159114292, 4.302512140927468, 9.99610293553897, 7.482413203872681, 3.534815478117813, 10.488816668434156, 7.242812195815706, 7.328342800785706, 5.505076763376644, 4.803549530719226, 5.875685672559412, 4.9733500673227855, 5.755952200345051, 4.832088418533625, 4.002114392994912, 3.9899727980121082, 7.905773664190996, 7.048715870576108, 6.016515184878076, 7.580574506965327, 6.947357742305049, 9.018666404842802, 7.006351954391287, 4.46506339296268, 12.20094007749519, 7.1400681595685525, 3.799119194571356, 10.183199650538082, 5.754051842686814, 7.261508400447562, 7.3442553227555445, 5.863449900579854, 3.403113254462003, 5.485051519034684, 9.508429005673891, 5.606576329310926, 5.268884881642422, 5.871542652220649]
6.56663821950658

Scoring generation toxicity: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:09<00:00,  5.19it/s]


0.12603398179169745 0.06
jajaja


Some weights of the model checkpoint at textattack/roberta-base-CoLA were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


jajaja2
writing outputs to  None


Scoring generation fluency: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 126.05it/s]

0.96
